In [4]:
import os
import pandas as pd
from mutagen.mp3 import MP3

# --- CONFIGURATION ---
METADATA_PATH = r"C:\Users\arday\Downloads\metadata.csv"
TRAIN_AUDIO_DIR = r"C:\Users\arday\Downloads\train_audios"
TEST_AUDIO_DIR = r"C:\Users\arday\Downloads\test_audios"
OUTPUT_FILE = r"C:\Users\arday\Desktop\UROP\english_audio_summary.csv"

def get_audio_features(mp3_path):
    """
    Reads an .mp3 file and extracts duration, sample rate, and channels.
    """
    if not os.path.exists(mp3_path):
        return None, None, None
        
    try:
        audio = MP3(mp3_path)
        duration_seconds = audio.info.length
        rate = audio.info.sample_rate
        channels = audio.info.channels
        
        return duration_seconds, rate, channels
    except Exception as e:
        print(f"Error reading audio {mp3_path}: {e}")
        return None, None, None

def main():
    """
    Main function to process English metadata and extract MP3 audio features.
    """
    print("Loading metadata...")
    if not os.path.exists(METADATA_PATH):
        print(f"ERROR: {METADATA_PATH} not found.")
        return

    df = pd.read_csv(METADATA_PATH)
    
    # Filter only English recordings
    df_english = df[df['language'].str.lower() == 'english'].copy()
    print(f"Found {len(df_english)} English records. Starting analysis...\n")
    
    results = []
    found_count = 0
    missing_count = 0
    
    # Process each English record
    for index, row in df_english.iterrows():
        uid = str(row['uid'])
        split_type = str(row.get('split', '')).lower()
        
        # Look for .mp3 files instead of .wav
        if split_type == 'test':
            audio_path = os.path.join(TEST_AUDIO_DIR, f"{uid}.mp3")
        else:
            audio_path = os.path.join(TRAIN_AUDIO_DIR, f"{uid}.mp3")
            
        # Extract Features
        duration, rate, channels = get_audio_features(audio_path)
        
        if duration is not None:
            found_count += 1
        else:
            missing_count += 1
            
        # Save results
        results.append({
            'UID': uid,
            'Diagnosis': row.get('diagnosis', ''),
            'Split': row.get('split', ''),
            'Audio_Duration_Seconds': duration,
            'Sample_Rate_Hz': rate,
            'Channels': channels
        })
        
    # Export to a new CSV file
    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_FILE, index=False)
    
    print("\n--- ANALYSIS COMPLETE ---")
    print(f"Total English records processed: {len(df_english)}")
    print(f"Audio files successfully read: {found_count}")
    print(f"Audio files missing/not found: {missing_count}")
    print(f"\nSaved results to: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Loading metadata...
Found 1655 English records. Starting analysis...


--- ANALYSIS COMPLETE ---
Total English records processed: 1655
Audio files successfully read: 1655
Audio files missing/not found: 0

Saved results to: C:\Users\arday\Desktop\UROP\english_audio_summary.csv


In [1]:
import os
import pandas as pd
import librosa
import numpy as np

# --- CONFIGURATION ---
METADATA_PATH = r"C:\Users\arday\Downloads\metadata.csv"
TRAIN_DIR = r"C:\Users\arday\Downloads\train_audios"
TEST_DIR = r"C:\Users\arday\Downloads\test_audios"
OUTPUT_FILE = r"C:\Users\arday\Desktop\UROP\advanced_audio_metrics.csv"

def analyze_audio(file_path):
    """
    Calculates RMS loudness and Silence proportion for an audio file.
    """
    if not os.path.exists(file_path):
        return None, None

    try:
        # Load audio (y is the audio array, sr is sample rate)
        y, sr = librosa.load(file_path, sr=None)
        
        # 1. Calculate RMS Loudness
        # This finds the average energy/volume of the track
        rms_array = librosa.feature.rms(y=y)
        mean_rms = float(np.mean(rms_array))
        
        # 2. Calculate Silence Proportion
        # librosa.effects.split finds the non-silent parts. 
        # top_db=20 means any sound 20dB below the peak is considered silence.
        non_silent_intervals = librosa.effects.split(y, top_db=20)
        
        non_silent_samples = 0
        for start, end in non_silent_intervals:
            non_silent_samples += (end - start)
            
        total_samples = len(y)
        silent_samples = total_samples - non_silent_samples
        
        # Calculate the percentage of silence
        silence_proportion = float(silent_samples / total_samples) if total_samples > 0 else 0.0
        
        return mean_rms, silence_proportion
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None, None

def main():
    print("Loading metadata...")
    df = pd.read_csv(METADATA_PATH)
    
    # Filter only English records
    df_english = df[df['language'].str.lower() == 'english'].copy()
    print(f"Found {len(df_english)} English files to process.")
    
    results = []
    
    for index, row in df_english.iterrows():
        uid = str(row['uid'])
        split_type = str(row.get('split', '')).lower()
        
        # Determine the correct folder based on split
        if split_type == 'test':
            audio_path = os.path.join(TEST_DIR, f"{uid}.mp3")
        else:
            audio_path = os.path.join(TRAIN_DIR, f"{uid}.mp3")
            
        print(f"Analyzing: {uid}.mp3")
        rms, silence = analyze_audio(audio_path)
        
        if rms is not None:
            results.append({
                'UID': uid,
                'RMS_Loudness': round(rms, 4),
                'Silence_Proportion': round(silence, 4)
            })
            
    # Export results to a new CSV file
    results_df = pd.DataFrame(results)
    results_df.to_csv(OUTPUT_FILE, index=False)
    print(f"Analysis Complete! Saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Loading metadata...
Found 1655 English files to process.
Analyzing: aaop.mp3
Analyzing: aazd.mp3
Analyzing: abgk.mp3
Analyzing: ablf.mp3
Analyzing: acdn.mp3
Analyzing: acqd.mp3
Analyzing: adoi.mp3
Analyzing: afop.mp3
Analyzing: afpc.mp3
Analyzing: afrv.mp3
Analyzing: afvm.mp3
Analyzing: agiy.mp3
Analyzing: agni.mp3
Analyzing: agok.mp3
Analyzing: agon.mp3
Analyzing: ahmw.mp3
Analyzing: ahyz.mp3
Analyzing: aijc.mp3
Analyzing: aikf.mp3
Analyzing: aikh.mp3
Analyzing: ajdc.mp3
Analyzing: ajef.mp3
Analyzing: akni.mp3
Analyzing: akqh.mp3
Analyzing: allm.mp3
Analyzing: alov.mp3
Analyzing: alrq.mp3
Analyzing: also.mp3
Analyzing: aluc.mp3
Analyzing: alyi.mp3
Analyzing: amdz.mp3
Analyzing: amhc.mp3
Analyzing: amhy.mp3
Analyzing: ammk.mp3
Analyzing: amsn.mp3
Analyzing: anek.mp3
Analyzing: anlg.mp3
Analyzing: anpg.mp3
Analyzing: ansa.mp3
Analyzing: aohw.mp3
Analyzing: aooz.mp3
Analyzing: aowd.mp3
Analyzing: apez.mp3
Analyzing: apns.mp3
Analyzing: apps.mp3
Analyzing: apyc.mp3
Analyzing: atia.mp3
Ana

In [1]:
import pandas as pd
import os
import shutil

# --- FILE PATHS ---
# Update these paths to match your computer
METADATA_PATH = r"C:\Users\arday\Downloads\metadata.csv"
TRAIN_AUDIO_DIR = r"C:\Users\arday\Downloads\train_audios"
OUTPUT_DIR = r"C:\Users\arday\Desktop\UROP\Random_Audio_Samples"

# Create the output folder if it does not exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- SELECTION RULES ---
# How many files we need from each corpus
target_counts = {
    'wls': 5,
    'pitt': 5,
    'vas': 3,
    'delaware': 3,
    'lu': 3,
    'hopkins': 3,
    'baycrest': 2,
    'depaul': 1
}

def select_random_audios():
    # Load metadata
    print("Reading metadata...")
    df = pd.read_csv(METADATA_PATH)
    
    # Keep only the 'train' files
    df_train = df[df['split'].str.lower() == 'train']
    
    print("Selecting files and copying to the new folder...")
    
    # Loop through our rules
    for corpus_name, needed_count in target_counts.items():
        # Find all train files for this specific corpus
        corpus_data = df_train[df_train['corpus'].str.lower() == corpus_name]
        
        # Check how many files actually exist
        available_files = len(corpus_data)
        
        # Pick the needed count, or all of them if we don't have enough
        pick_count = min(needed_count, available_files)
        
        if pick_count == 0:
            print(f"No files found for corpus: {corpus_name}")
            continue
            
        # Select random rows
        random_picks = corpus_data.sample(n=pick_count)
        
        # Copy each selected file to the new folder
        for index, row in random_picks.iterrows():
            uid = str(row['uid'])
            # Assuming your files are .mp3. Change to .wav if necessary.
            filename = f"{uid}.mp3" 
            
            source_path = os.path.join(TRAIN_AUDIO_DIR, filename)
            # Add corpus name to the file so you know what you are listening to
            new_filename = f"{corpus_name}_{filename}"
            target_path = os.path.join(OUTPUT_DIR, new_filename)
            
            # Check if file exists on your computer before copying
            if os.path.exists(source_path):
                shutil.copy2(source_path, target_path)
                print(f"Copied: {new_filename}")
            else:
                print(f"Missing audio file: {source_path}")

    print(f"\nSuccess! All random samples are ready inside: {OUTPUT_DIR}")

# Run the script
if __name__ == "__main__":
    select_random_audios()

Reading metadata...
Selecting files and copying to the new folder...
Copied: wls_fnas.mp3
Copied: wls_wyub.mp3
Copied: wls_vjuo.mp3
Copied: wls_dcuv.mp3
Copied: wls_kkbu.mp3
Copied: pitt_heut.mp3
Copied: pitt_xsrh.mp3
Copied: pitt_bxkp.mp3
Copied: pitt_xdjx.mp3
Copied: pitt_zjso.mp3
Copied: vas_ojvu.mp3
Copied: vas_cvxa.mp3
Copied: vas_vpxq.mp3
Copied: delaware_zaok.mp3
Copied: delaware_qhjd.mp3
Copied: delaware_riyp.mp3
Copied: lu_gimv.mp3
Copied: lu_xfag.mp3
Copied: lu_vjci.mp3
Copied: hopkins_kfdc.mp3
Copied: hopkins_qgfg.mp3
Copied: hopkins_ghjh.mp3
Copied: baycrest_couz.mp3
Copied: baycrest_epon.mp3
Copied: depaul_hotv.mp3

Success! All random samples are ready inside: C:\Users\arday\Desktop\UROP\Random_Audio_Samples


In [6]:
import os
import glob
import json
import re
import string
import difflib
import spacy

# --- CONFIGURATION ---
# Sadece klasör yollarını veriyoruz, arama ifadelerini sildik.
GOLD_DIR = r"C:\Users\arday\Desktop\UROP\Project_Codes\ADReSSo Project\data\gold_standard"
WHISPER_DIR = r"C:\Users\arday\Desktop\UROP\Project_Codes\ADReSSo Project\data\transcripts"
OUTPUT_REPORT = r"C:\Users\arday\Desktop\UROP\Project_Codes\ADReSSo Project\data\Wiki_Report.md"

HESITATIONS = ["uh", "um", "ah", "er", "hmm", "well", "hes", "eee", "aaa", "iii", "ehm"]
CONTENT_TAGS = ["NOUN", "VERB", "ADJ", "ADV", "PROPN"]

def clean_word(word):
    """Removes punctuation and makes text lowercase."""
    return word.translate(str.maketrans('', '', string.punctuation)).lower().strip()

def main():
    print("Loading spaCy NLP engine...")
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        print("ERROR: Run 'python -m spacy download en_core_web_sm' first.")
        return

    # Iterate through Whisper outputs (JSONs) instead of Gold Standard (TXTs)
    json_files = glob.glob(os.path.join(WHISPER_DIR, "*.json"))
    if not json_files:
        print("No Whisper JSON files found.")
        return

    results = []

    for json_path in json_files:
        file_name = os.path.basename(json_path)
        base_name = os.path.splitext(file_name)[0]

        # 1. Read Whisper JSON
        whisper_words = []
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for item in data.get("words", []):
                whisper_words.append(item.get("word", ""))

        # 2. Try to Read Gold Standard TXT
        txt_path = os.path.join(GOLD_DIR, f"{base_name}.txt")
        gold_words = []
        has_gold = False
        
        if os.path.exists(txt_path):
            with open(txt_path, 'r', encoding='utf-8') as f:
                content = f.read().strip()
                if content:
                    has_gold = True
                    parts = re.split(r'\t+', content, maxsplit=1)
                    gold_text = parts[1].strip() if len(parts) == 2 else content
                    gold_words = gold_text.split()

        # 3. Clean Words
        gold_clean = [clean_word(w) for w in gold_words]
        whisper_clean = [clean_word(w) for w in whisper_words]

        # 4. Handle Missing/Empty Gold Standard Files
        if not has_gold:
            if len(whisper_clean) == 0:
                results.append({
                    "name": base_name,
                    "wer": 0.0,
                    "status": "No Gold Standard - Model correctly stayed silent."

_IncompleteInputError: incomplete input (184109192.py, line 73)

In [1]:
import os
import glob
import pandas as pd
import string

# --- CONFIGURATION (Dynamic Relative Paths) ---

# 1. Find the exact folder where this Python script is located
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# 2. Set Paths
INPUT_DIR = os.path.join(BASE_DIR, "..", "..", "data", "alignment_reports")
OUTPUT_DIR = os.path.join(BASE_DIR, "..", "..", "data", "verticalized_data")

# --- HESITATION MARKS (Expanded List) ---
# A comprehensive set of typical hesitation, filler, and breath words generated by Whisper
HESITATION_MARKS = {
    # A, E, I (Vowel variations)
    "ah", "ahh", "ahhh", "aaa",
    "eh", "ehh", "ehm", "ehmm", "eee",
    "ih", "ihh", "iii",
    "oh", "ooh", "ooo",
    
    # U, M (Mumbling / Thinking variations)
    "uh", "uhh", "uhm", "uhhm", "uuu",
    "um", "umm", "ummm",
    "mm", "mmm", "mmmm",
    "hm", "hmm", "hmmm", "hmmmm",
    "mhm", "mhmm",
    
    # Er / Erm variations
    "er", "err", "errr", "erm", "errm",
    
    # Breaths, Clicks, and other fillers
    "tsk", "pff", "pfff", "pffff",
    "huh", "huhh", "aha", "shh", "shhh", "ss"
}

def clean_text_for_comparison(word):
    """
    Removes punctuation and normalizes casing to accurately determine error types.
    """
    if pd.isna(word): 
        return ""
    text = str(word).lower().strip()
    text = text.translate(str.maketrans('', '', string.punctuation))
    if text == "unk": 
        return ""
    return text


def get_alignment_metadata(row):
    """
    Compares the Gold Standard and Whisper words to assign a basic Error Type.
    """
    gold = clean_text_for_comparison(row.get('Gold_Word', ''))
    whisper = clean_text_for_comparison(row.get('Whisper_Word', ''))
    
    # --- HESITATION MATCHING LOGIC ---
    # If Gold Standard is annotated as "HES" and Whisper produced a hesitation mark, count as MATCH
    if gold == "hes" and whisper in HESITATION_MARKS:
        return "MATCH"
    # --------------------------------------
        
    if gold == "" and whisper != "": 
        return "INSERTION"
    elif gold != "" and whisper == "": 
        return "DELETION"
    elif gold == whisper: 
        return "MATCH"
    else: 
        return "SUBSTITUTION"

def main():
    """
    Main pipeline function for Step 4.
    Reads aligned CSVs, adds alignment metadata, and creates verticalized datasets.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    if not os.path.exists(INPUT_DIR):
        print(f"ERROR: Cannot find input directory at {INPUT_DIR}")
        return

    csv_files = glob.glob(os.path.join(INPUT_DIR, "**", "*.csv"), recursive=True)
    print(f"Found {len(csv_files)} aligned files. Starting verticalization...\n")
    
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        
        # Keep the exact audio name without the extra extensions
        audio_name = file_name.replace("_aligned.csv", "").replace(".csv", "")
        
        try:
            df = pd.read_csv(file_path)
            
            # Clean up: Remove audio ID from the first row if present
            first_gold = str(df.loc[0, 'Gold_Word']) if 'Gold_Word' in df.columns else ""
            if audio_name in first_gold or "adrs" in first_gold:
                df = df.iloc[1:].reset_index(drop=True)
            
            # Add Alignment Metadata
            df['Error_Type'] = df.apply(get_alignment_metadata, axis=1)
            
            # Reorder columns
            cols = ['Gold_Word', 'Whisper_Word', 'Start_Time', 'End_Time', 'Error_Type']
            df_vertical = df[[c for c in cols if c in df.columns]]
            
            # Save the verticalized dataset
            out_path = os.path.join(OUTPUT_DIR, f"{audio_name}_verticalized.csv")
            df_vertical.to_csv(out_path, index=False, encoding='utf-8')
            
            print(f"SUCCESS - Verticalized: {audio_name}_verticalized.csv")
            
        except Exception as e:
            print(f"ERROR - Could not process {file_name}: {e}")

    print("\nVerticalization pipeline finished.")

if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined

In [5]:
import os
import glob
import pandas as pd
import spacy
import warnings

# Hide unnecessary warnings
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
INPUT_DIR = os.path.join(BASE_DIR, "..", "..", "data", "verticalized_data")
OUTPUT_CSV = os.path.join(BASE_DIR, "..", "..", "data", "Ablation_Summary.csv")

# --- NLP Settings ---
HESITATIONS = ["uh", "um", "ah", "er", "hmm", "well", "hes"]
FUNCTION_TAGS = ["DET", "ADP", "AUX", "PRON", "CCONJ", "SCONJ", "PART"]
CONTENT_TAGS = ["NOUN", "VERB", "ADJ", "ADV", "PROPN"]


def get_linguistic_category(row, nlp_engine):
    """
    Applies NLP to categorize the linguistic impact of the word.
    """
    error_type = row.get('Error_Type', '')
    gold = str(row.get('Gold_Word', '')).lower()
    whisper = str(row.get('Whisper_Word', '')).lower()
    
    if error_type == "MATCH":
        return "Correctly_Recognized"
        
    target = gold if error_type in ["DELETION", "SUBSTITUTION"] else whisper
    
    if not target or target == "nan": 
        return "Other"
        
    if target in HESITATIONS or whisper in HESITATIONS: 
        return "Hesitation_Marker"
        
    doc = nlp_engine(target)
    target_lemma = doc[0].lemma_
    target_pos = doc[0].pos_
    
    if error_type == "SUBSTITUTION":
        doc_w = nlp_engine(whisper)
        whisper_lemma = doc_w[0].lemma_ if len(doc_w) > 0 else ""
        if target_lemma == whisper_lemma and target_lemma != "":
            return "Morphological_Deviation"
            
    if target_pos in FUNCTION_TAGS: 
        return "Function_Word"
    if target_pos in CONTENT_TAGS: 
        return "Content_Word"
        
    return "Other"


def main():
    if not os.path.exists(INPUT_DIR):
        print(f"ERROR: Cannot find input directory at {INPUT_DIR}")
        return

    print("Loading spaCy NLP engine (en_core_web_sm)...")
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        print("ERROR: spaCy model not found. Please run: python -m spacy download en_core_web_sm")
        return

    csv_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
    if not csv_files:
        print(f"No verticalized CSV files found in {INPUT_DIR}.")
        return

    print(f"Starting Error Analysis for {len(csv_files)} files...\n")
    
    summary_data = [] 
    
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        audio_name = file_name.replace("_verticalized.csv", "")
        
        # 1. Skip completely empty files to prevent EmptyDataError
        if os.path.getsize(file_path) == 0:
            print(f"Skipping {file_name}: File is completely empty.")
            continue
            
        try:
            # 2. Read the valid CSV file
            df = pd.read_csv(file_path)
            
            # 3. Check if dataframe is empty or missing required columns
            if df.empty or 'Error_Type' not in df.columns:
                print(f"Skipping {file_name}: Invalid format or no data rows.")
                continue
            
            df['Linguistic_Category'] = df.apply(lambda row: get_linguistic_category(row, nlp), axis=1)
            
            sub_count = len(df[df['Error_Type'] == 'SUBSTITUTION'])
            del_count = len(df[df['Error_Type'] == 'DELETION'])
            ins_count = len(df[df['Error_Type'] == 'INSERTION'])
            
            reference_words = len(df[df['Error_Type'].isin(['MATCH', 'DELETION', 'SUBSTITUTION'])])
            total_errors = sub_count + del_count + ins_count
            
            wer = (total_errors / reference_words) * 100 if reference_words > 0 else 0
            
            summary_data.append({
                "File_Name": audio_name,
                "WER (%)": round(wer, 2),
                "Substitutions": sub_count,
                "Deletions": del_count,
                "Insertions": ins_count
            })
            
        # Catch any unexpected pandas empty data errors just in case
        except pd.errors.EmptyDataError:
            print(f"Skipping {file_name}: pandas found no columns to parse.")
        except Exception as e:
            print(f"File {file_name} could not be processed. Error: {e}")

    if not summary_data:
        print("No valid data found to summarize.")
        return

    # Create the final dataframe and save to CSV
    df_summary = pd.DataFrame(summary_data)
    df_summary.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\n[SUCCESS] Pipeline Complete!")
    print(f"Results saved to: {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

NameError: name '__file__' is not defined

In [16]:
import os
import glob
import pandas as pd

def process_csv_files():
    """
    Reads all CSV files from a user-provided folder, counts the errors, 
    and saves a summary table in the same folder.
    """
    # 1. Ask the user to paste the folder path
    print("Example path: C:\\Users\\Name\\Desktop\\verticalized_data")
    INPUT_DIR = r"C:\Users\arday\Desktop\UROP\Project_Codes\ADReSSo_Project\data\verticalized_data"
    
    # Check if the provided folder actually exists
    if not os.path.isdir(INPUT_DIR):
        print("\nError: The folder path you provided does not exist.")
        print("Please check the path and try again.")
        return

    # 2. Set the output file to be saved in the exact same folder
    OUTPUT_CSV = os.path.join(INPUT_DIR, "Ablation_Summary.csv")
    
    # Find all .csv files in the input folder
    csv_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
    
    # Prevent the script from reading its own summary file if you run it twice
    csv_files = [f for f in csv_files if "Ablation_Summary.csv" not in f]
    
    if not csv_files:
        print("\nNo CSV files found in:", INPUT_DIR)
        return
        
    summary_list = []
    print(f"\nFound {len(csv_files)} CSV files. Processing...")
    
    # 3. Process each file
    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        clean_name = file_name.replace("_verticalized.csv", "").replace(".csv", "")
        
        # Default values
        sub = 0
        del_count = 0
        ins = 0
        wer = 0.0
        
        # Check if the file is NOT empty
        if os.path.getsize(file_path) > 0:
            try:
                df = pd.read_csv(file_path)
                
                if 'Error_Type' in df.columns:
                    sub = len(df[df['Error_Type'] == 'SUBSTITUTION'])
                    del_count = len(df[df['Error_Type'] == 'DELETION'])
                    ins = len(df[df['Error_Type'] == 'INSERTION'])
                    match_count = len(df[df['Error_Type'] == 'MATCH'])
                    
                    reference_words = match_count + sub + del_count
                    total_errors = sub + del_count + ins
                    
                    if reference_words > 0:
                        wer = (total_errors / reference_words) * 100
                        
            except pd.errors.EmptyDataError:
                pass
            except Exception as e:
                print(f"Could not read {file_name}. Error: {e}")
        
        # Append data for this file
        summary_list.append({
            "File_Name": clean_name,
            "WER (%)": round(wer, 2),
            "Substitutions": sub,
            "Deletions": del_count,
            "Insertions": ins
        })
        
    # 4. Save the final table
    final_table = pd.DataFrame(summary_list)
    final_table.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\nSuccess! Processed {len(summary_list)} files.")
    print(f"The summary table is saved at: {OUTPUT_CSV}")

if __name__ == "__main__":
    process_csv_files()

Example path: C:\Users\Name\Desktop\verticalized_data

Found 27 CSV files. Processing...

Success! Processed 27 files.
The summary table is saved at: C:\Users\arday\Desktop\UROP\Project_Codes\ADReSSo_Project\data\verticalized_data\Ablation_Summary.csv


In [1]:
import os
import shutil
import pandas as pd

# 1. Define folder and file paths
metadata_file = r"C:\Users\arday\Downloads\metadata.csv"

source_train_dir = r"C:\Users\arday\Downloads\train_audios"
source_test_dir = r"C:\Users\arday\Downloads\test_audios"

output_train_dir = r"C:\Users\arday\Desktop\aaaa\train"
output_test_dir = r"C:\Users\arday\Desktop\aaaa\test"

# 2. Create the new folders if they do not exist
os.makedirs(output_train_dir, exist_ok=True)
os.makedirs(output_test_dir, exist_ok=True)

# 3. Read metadata and get a list of English filenames
# Note: Update 'language' and 'filename' to match your actual CSV column names
df = pd.read_csv(metadata_file)
english_data = df[df['language'] == 'en'] 
english_filenames = english_data['filename'].tolist()

# 4. Create a function to copy files
def filter_and_copy(source_dir, output_dir, valid_files):
    # Check every file in the source folder
    for current_file in os.listdir(source_dir):
        # If the file is in our English list, copy it
        if current_file in valid_files:
            source_path = os.path.join(source_dir, current_file)
            output_path = os.path.join(output_dir, current_file)
            
            shutil.copy2(source_path, output_path)

# 5. Run the function for both train and test folders
print("Processing train files...")
filter_and_copy(source_train_dir, output_train_dir, english_filenames)

print("Processing test files...")
filter_and_copy(source_test_dir, output_test_dir, english_filenames)

print("All English files copied successfully!")

KeyError: 'filename'

In [2]:
import os
import shutil
import pandas as pd
from pathlib import Path

# 1. Dosya ve klasör yollarını belirleyin
metadata_file = r"C:\Users\arday\Downloads\metadata.csv"

source_train_dir = r"C:\Users\arday\Downloads\train_audios"
source_test_dir = r"C:\Users\arday\Downloads\test_audios"

output_train_dir = r"C:\Users\arday\Desktop\aaaa\train"
output_test_dir = r"C:\Users\arday\Desktop\aaaa\test"

# 2. Çıktı klasörlerini oluşturun
os.makedirs(output_train_dir, exist_ok=True)
os.makedirs(output_test_dir, exist_ok=True)

# 3. Metadata'yı okuyun ve İngilizce (english) olan 'uid' listesini alın
df = pd.read_csv(metadata_file)
# 'language' sütunundaki değerin 'english' olduğu satırları filtreliyoruz
english_data = df[df['language'] == 'english']
# Dosyaları eşleştirmek için 'uid' sütununu listeye çeviriyoruz
english_uids = set(english_data['uid'].astype(str).tolist())

# 4. Dosyaları filtreleyip kopyalayan fonksiyon
def filter_and_copy(source_dir, output_dir, valid_uids):
    if not os.path.exists(source_dir):
        print(f"Uyarı: {source_dir} klasörü bulunamadı.")
        return

    copied_count = 0
    # Kaynak klasördeki tüm dosyaları kontrol et
    for current_file in os.listdir(source_dir):
        # Dosya uzantısını ayırarak sadece kök ismini (uid) alıyoruz (örn: 'aaop.wav' -> 'aaop')
        file_stem = Path(current_file).stem
        
        # Eğer dosya ismi İngilizce UID listemizde varsa kopyala
        if file_stem in valid_uids:
            source_path = os.path.join(source_dir, current_file)
            output_path = os.path.join(output_dir, current_file)
            
            shutil.copy2(source_path, output_path)
            copied_count += 1
            
    print(f"{source_dir} klasöründen {copied_count} adet İngilizce dosya kopyalandı.")

# 5. İşlemi train ve test klasörleri için çalıştırın
print("Train dosyaları işleniyor...")
filter_and_copy(source_train_dir, output_train_dir, english_uids)

print("Test dosyaları işleniyor...")
filter_and_copy(source_test_dir, output_test_dir, english_uids)

print("İşlem tamamlandı!")

Train dosyaları işleniyor...
C:\Users\arday\Downloads\train_audios klasöründen 1331 adet İngilizce dosya kopyalandı.
Test dosyaları işleniyor...
C:\Users\arday\Downloads\test_audios klasöründen 324 adet İngilizce dosya kopyalandı.
İşlem tamamlandı!


In [2]:
import os
import shutil
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment

# ── Paths ──────────────────────────────────────────────
METADATA     = r"C:\Users\arday\Downloads\metadata.csv"
TRAIN_FOLDER = r"C:\Users\arday\Downloads\train_audios"
OUTPUT       = r"C:\Users\arday\Desktop\pilot_dataset_2"
# ───────────────────────────────────────────────────────

df = pd.read_csv(METADATA)
df['age'] = pd.to_numeric(df['age'], errors='coerce')

eng = df[(df['language'] == 'english') & (df['split'] == 'train') & df['age'].notna()]
ctrl = eng[eng['diagnosis'] == 'Control'].reset_index(drop=True)
ad   = eng[eng['diagnosis'].isin(['ProbableAD', 'AD'])].reset_index(drop=True)

print(f"Pool — Control: {len(ctrl)}  AD: {len(ad)}")

sel_ctrl, sel_ad, pairs = [], [], []

for gender in ['female', 'male']:
    cg = ctrl[ctrl['gender'] == gender].reset_index(drop=True)
    ag = ad[ad['gender'] == gender].reset_index(drop=True)
    if len(cg) == 0 or len(ag) == 0:
        continue
    cost = np.abs(cg['age'].values[:, None] - ag['age'].values[None, :])
    ri, ci = linear_sum_assignment(cost)
    ranked = sorted(zip(ri, ci, cost[ri, ci]), key=lambda x: x[2])
    for r, c, diff in ranked[:25]:
        sel_ctrl.append(cg.iloc[r].to_dict())
        sel_ad.append(ag.iloc[c].to_dict())
        pairs.append({'ctrl': cg.iloc[r]['uid'], 'ctrl_age': int(cg.iloc[r]['age']),
                      'ad':   ag.iloc[c]['uid'], 'ad_age':   int(ag.iloc[c]['age']),
                      'diff': int(diff), 'gender': gender})

df_ctrl = pd.DataFrame(sel_ctrl)
df_ad   = pd.DataFrame(sel_ad)
df_pairs = pd.DataFrame(pairs)

print(f"\nSelected — Control: {len(df_ctrl)}  AD: {len(df_ad)}")
print(f"Control  age: {df_ctrl['age'].mean():.1f} ± {df_ctrl['age'].std():.1f}  gender: {df_ctrl['gender'].value_counts().to_dict()}")
print(f"AD       age: {df_ad['age'].mean():.1f} ± {df_ad['age'].std():.1f}  gender: {df_ad['gender'].value_counts().to_dict()}")
print(f"Mean age diff per pair: {df_pairs['diff'].mean():.1f} yrs")
print(f"\nPairs:\n{df_pairs.to_string(index=False)}")

os.makedirs(os.path.join(OUTPUT, 'control'),   exist_ok=True)
os.makedirs(os.path.join(OUTPUT, 'alzheimer'), exist_ok=True)

def copy_audio(uid, dest):
    for root, _, files in os.walk(TRAIN_FOLDER):
        for f in files:
            if f.startswith(uid) and f.endswith(('.wav', '.mp3', '.flac', '.m4a')):
                shutil.copy2(os.path.join(root, f), os.path.join(dest, f))
                return True
    print(f"  [!] Not found: {uid}")
    return False

for row in sel_ctrl: copy_audio(row['uid'], os.path.join(OUTPUT, 'control'))
for row in sel_ad:   copy_audio(row['uid'], os.path.join(OUTPUT, 'alzheimer'))

df_ctrl.to_csv(os.path.join(OUTPUT, 'control_metadata.csv'),   index=False)
df_ad.to_csv(os.path.join(OUTPUT,   'alzheimer_metadata.csv'), index=False)
df_pairs.to_csv(os.path.join(OUTPUT, 'pairs.csv'),             index=False)

print(f"\nDone → {OUTPUT}/")

Pool — Control: 760  AD: 423

Selected — Control: 50  AD: 50
Control  age: 74.2 ± 8.6  gender: {'female': 25, 'male': 25}
AD       age: 74.2 ± 8.6  gender: {'female': 25, 'male': 25}
Mean age diff per pair: 0.0 yrs

Pairs:
ctrl  ctrl_age   ad  ad_age  diff gender
afop        67 aikf      67     0 female
afrv        80 acqd      80     0 female
ahyz        71 xwva      71     0 female
alov        47 eaze      47     0 female
also        81 amsn      81     0 female
aluc        83 dbsd      83     0 female
amhy        82 crpv      82     0 female
apez        79 jjpd      79     0 female
apps        81 ansa      81     0 female
atmc        74 dwqi      74     0 female
aulf        75 akni      75     0 female
awxj        70 cevp      70     0 female
ayqu        78 bcap      78     0 female
bdtr        81 bbki      81     0 female
bhms        81 ctie      81     0 female
bkju        81 dokm      81     0 female
blde        61 kflj      61     0 female
bldi        74 lgve      74     0 femal

In [3]:
"""
43 Acoustic Biomarker Extractor + Visualization
------------------------------------------------
pip install praat-parselmouth librosa scipy pandas matplotlib

Usage: python extract_features.py
"""

import os, glob
import numpy as np
import pandas as pd
import librosa
import parselmouth
from parselmouth.praat import call
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

# ── Paths ───────────────────────────────────────────────
CONTROL_DIR   = r"C:\Users\arday\Desktop\pilot_dataset_2\control"
ALZHEIMER_DIR = r"C:\Users\arday\Desktop\pilot_dataset_2\alzheimer"
OUTPUT_CSV    = r"C:\Users\arday\Desktop\pilot_dataset_2\features_40.csv"
PLOTS_DIR     = r"C:\Users\arday\Desktop\pilot_dataset_2\feature_plots"
# ────────────────────────────────────────────────────────

AUDIO_EXTS = ('.wav', '.mp3', '.flac', '.m4a')


def stats6(arr):
    """mean, std, min, max, kurtosis, skewness"""
    arr = arr[~np.isnan(arr)]
    if len(arr) < 2:
        return [np.nan] * 6
    return [np.mean(arr), np.std(arr), np.min(arr), np.max(arr),
            float(stats.kurtosis(arr)), float(stats.skew(arr))]


# ── Feature groups ──────────────────────────────────────

def f0_features(snd):
    pitch = snd.to_pitch_ac(pitch_floor=75, pitch_ceiling=600)
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]
    if len(f0) < 2:
        return {k: np.nan for k in
                ['f0_mean','f0_std','f0_min','f0_max','f0_kurtosis','f0_skewness','f0_std_norm']}
    m, s, mn, mx, ku, sk = stats6(f0)
    return {'f0_mean': m, 'f0_std': s, 'f0_min': mn, 'f0_max': mx,
            'f0_kurtosis': ku, 'f0_skewness': sk,
            'f0_std_norm': s / m if m > 0 else np.nan}


def intensity_features(snd):
    intensity = snd.to_intensity(minimum_pitch=75)
    vals = intensity.values.flatten()
    m, s, mn, mx, ku, sk = stats6(vals)
    return {'intensity_mean': m, 'intensity_std': s, 'intensity_min': mn,
            'intensity_max': mx, 'intensity_kurtosis': ku, 'intensity_skewness': sk}


def energy_features(y):
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=256)[0]
    m, s, mn, mx, ku, sk = stats6(rms)
    return {'energy_mean': m, 'energy_std': s, 'energy_min': mn,
            'energy_max': mx, 'energy_kurtosis': ku, 'energy_skewness': sk}


def pause_features(snd):
    total_dur = snd.get_total_duration()
    intensity = snd.to_intensity(minimum_pitch=75)
    tg = call(intensity, "To TextGrid (silences)", -25, 0.1, 0.05, "silence", "speech")

    n = call(tg, "Get number of intervals", 1)
    pauses, speech_segs = [], []
    for i in range(1, n + 1):
        label = call(tg, "Get label of interval", 1, i)
        dur = (call(tg, "Get end time of interval", 1, i) -
               call(tg, "Get start time of interval", 1, i))
        (pauses if label == "silence" else speech_segs).append(dur)

    p = np.array(pauses)   if pauses      else np.array([0.0])
    s = np.array(speech_segs) if speech_segs else np.array([0.0])
    sp = s.sum()
    ipi = s[1:-1] if len(s) > 2 else np.array([0.0])

    return {
        'mean_pause_duration':          float(np.mean(p)),
        'total_speech_time':            float(sp),
        'total_pause_time':             float(p.sum()),
        'pause_percentage':             float(p.sum() / total_dur * 100),
        'silence_to_speech_ratio':      float(p.sum() / sp) if sp > 0 else 0.0,
        'pause_variability':            float(np.std(p)),
        'number_of_pauses':             len(pauses),
        'number_of_speech_segments':    len(speech_segs),
        'max_pause_duration':           float(np.max(p)),
        'min_pause_duration':           float(np.min(p)),
        'pause_rate':                   len(pauses) / total_dur,
        'mean_speech_segment_duration': float(np.mean(s)),
        'std_speech_segment_duration':  float(np.std(s)),
        'mean_inter_pause_interval':    float(np.mean(ipi)),
        'std_inter_pause_interval':     float(np.std(ipi)),
        'long_pause_count':             int(np.sum(p > 0.5)),
        'short_pause_count':            int(np.sum(p <= 0.5)),
        'pause_duration_skewness':      float(stats.skew(p)) if len(p) > 2 else 0.0,
    }


def articulation_features(snd):
    total_dur = snd.get_total_duration()

    # Voiced duration → approximate syllable count
    pitch = snd.to_pitch_ac(pitch_floor=75, pitch_ceiling=600)
    f0 = pitch.selected_array['frequency']
    voiced_dur = np.sum(f0 > 0) * pitch.time_step
    n_syl = max(voiced_dur / 0.15, 1)  # ~150ms per syllable average

    # Speech time (excl. pauses)
    intensity = snd.to_intensity(minimum_pitch=75)
    tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
    n = call(tg, "Get number of intervals", 1)
    speech_time = sum(
        call(tg, "Get end time of interval", 1, i) - call(tg, "Get start time of interval", 1, i)
        for i in range(1, n + 1)
        if call(tg, "Get label of interval", 1, i) == "speech"
    )

    return {
        'speech_rate':           n_syl / total_dur,
        'articulation_rate':     n_syl / speech_time if speech_time > 0 else np.nan,
        'avg_syllable_duration': speech_time / n_syl if n_syl > 0 else np.nan,
    }


def formant_features(snd):
    fmt = snd.to_formant_burg(max_number_of_formants=5, maximum_formant=5500,
                               window_length=0.025, pre_emphasis_from=50)
    t = fmt.xs()
    f1 = np.array([call(fmt, "Get value at time", 1, ti, 'Hertz', 'Linear') for ti in t])
    f2 = np.array([call(fmt, "Get value at time", 2, ti, 'Hertz', 'Linear') for ti in t])
    f1, f2 = f1[~np.isnan(f1)], f2[~np.isnan(f2)]
    disp = np.mean(f2) - np.mean(f1) if len(f1) and len(f2) else np.nan
    vtl = 34300 / (2 * disp) if disp and disp > 0 else np.nan
    return {'f1_std': np.std(f1) if len(f1) > 1 else np.nan,
            'f2_std': np.std(f2) if len(f2) > 1 else np.nan,
            'vocal_tract_length': vtl}


# ── Main extraction ─────────────────────────────────────

def extract_all(filepath):
    y, _ = librosa.load(filepath, sr=16000, mono=True)
    snd   = parselmouth.Sound(filepath)
    row = {}
    row.update(f0_features(snd))
    row.update(intensity_features(snd))
    row.update(energy_features(y))
    row.update(pause_features(snd))
    row.update(articulation_features(snd))
    row.update(formant_features(snd))
    return row


def load_folder(folder, label):
    rows = []
    files = [f for f in sorted(glob.glob(os.path.join(folder, '*')))
             if f.endswith(AUDIO_EXTS)]
    for fp in files:
        uid = Path(fp).stem
        print(f"  [{label}] {uid}")
        try:
            row = extract_all(fp)
            row['uid'] = uid
            row['group'] = label
            rows.append(row)
        except Exception as e:
            print(f"    ERROR: {e}")
    return rows


print("Extracting features...")
rows = load_folder(CONTROL_DIR, 'Control') + load_folder(ALZHEIMER_DIR, 'Alzheimer')
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved → {OUTPUT_CSV}  ({len(df)} rows)")


# ── Visualization ───────────────────────────────────────

FEATURE_COLS = [c for c in df.columns if c not in ('uid', 'group')]
GROUPS  = ['Control', 'Alzheimer']
COLORS  = {'Control': '#60a5fa', 'Alzheimer': '#f87171'}
os.makedirs(PLOTS_DIR, exist_ok=True)

n_cols = 6
n_rows = -(-len(FEATURE_COLS) // n_cols)   # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 3.2))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    ax.set_facecolor('#111827')
    ax.spines[['top', 'right']].set_visible(False)
    for spine in ax.spines.values():
        spine.set_color('#1e293b')

    data = [df[df['group'] == g][feat].dropna().values for g in GROUPS]

    bp = ax.boxplot(data, patch_artist=True, widths=0.4,
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(color='#475569'),
                    capprops=dict(color='#475569'),
                    flierprops=dict(marker='o', markerfacecolor='#475569', markersize=2))
    for patch, g in zip(bp['boxes'], GROUPS):
        patch.set_facecolor(COLORS[g])
        patch.set_alpha(0.65)

    # Individual points with jitter
    for j, (vals, g) in enumerate(zip(data, GROUPS)):
        jitter = np.random.uniform(-0.12, 0.12, len(vals))
        ax.scatter(np.full(len(vals), j + 1) + jitter, vals,
                   color=COLORS[g], alpha=0.55, s=14, zorder=5)

    # Stats annotation
    if len(data[0]) > 1 and len(data[1]) > 1:
        pooled = np.sqrt((np.var(data[0]) + np.var(data[1])) / 2)
        d = (np.mean(data[1]) - np.mean(data[0])) / pooled if pooled > 0 else 0
        _, p = stats.ttest_ind(data[0], data[1], equal_var=False)
        col = '#fbbf24' if p < 0.05 else '#94a3b8'
        ax.set_title(f"{feat}\nd={d:.2f}  p={p:.3f}", fontsize=6.5, color=col, pad=4)
    else:
        ax.set_title(feat, fontsize=6.5, color='#94a3b8', pad=4)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Ctrl', 'AD'], fontsize=7, color='#94a3b8')
    ax.tick_params(axis='y', labelsize=6, colors='#64748b')

for i in range(len(FEATURE_COLS), len(axes)):
    axes[i].set_visible(False)

legend = [Patch(facecolor=COLORS[g], label=g) for g in GROUPS]
fig.legend(handles=legend, loc='lower right', fontsize=9,
           facecolor='#111827', edgecolor='#1e293b', labelcolor='white')

fig.suptitle('Acoustic Biomarkers — Control vs Alzheimer (n=40)',
             fontsize=13, color='white', y=1.005)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'all_features.png')
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.close()
print(f"Plot saved → {out}")
print("\nDone.")

Extracting features...
  [Control] abgk
  [Control] ablf
  [Control] afop
  [Control] afpc
  [Control] afrv
  [Control] agok
  [Control] ahmw
  [Control] ahyz
  [Control] ajdc
  [Control] alov
  [Control] alrq
  [Control] also
  [Control] aluc
  [Control] amhy
  [Control] anpg
  [Control] aohw
  [Control] apez
  [Control] apns
  [Control] apps
  [Control] apyc
  [Control] atmc
  [Control] aulf
  [Control] avvv
  [Control] awch
  [Control] awxj
  [Control] ayqu
  [Control] ayro
  [Control] bdtr
  [Control] bguv
  [Control] bhms
  [Control] bijy
  [Control] bjzd
  [Control] bkju
  [Control] blde
  [Control] bldi
  [Control] blyu
  [Control] bmok
  [Control] bnib
  [Control] bnxv
  [Control] bpdx
  [Control] bpom
  [Control] brou
  [Control] brrd
  [Control] bufm
  [Control] bumu
  [Control] bxog
  [Control] bymf
  [Control] caur
  [Control] ccbt
  [Control] ceqd
  [Alzheimer] acqd
  [Alzheimer] aikf
  [Alzheimer] akni
  [Alzheimer] amsn
  [Alzheimer] ansa
  [Alzheimer] avdc
  [Alzheimer]

In [ ]:
"""
43 Acoustic Biomarker Extractor + Visualization
------------------------------------------------
pip install praat-parselmouth librosa scipy pandas matplotlib webrtcvad-wheels

Usage: python extract_features.py
"""

import os, glob
from itertools import groupby
import numpy as np
import pandas as pd
import librosa
import parselmouth
from parselmouth.praat import call
from scipy import stats
import webrtcvad
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from pathlib import Path

# ── Paths ───────────────────────────────────────────────
CONTROL_DIR   = r"C:\Users\arday\Desktop\pilot_dataset_2\control"
ALZHEIMER_DIR = r"C:\Users\arday\Desktop\pilot_dataset_2\alzheimer"
OUTPUT_CSV    = r"C:\Users\arday\Desktop\pilot_dataset_2\features_100.csv"
PLOTS_DIR     = r"C:\Users\arday\Desktop\pilot_dataset_2\feature_plots"
# ────────────────────────────────────────────────────────

AUDIO_EXTS = ('.wav', '.mp3', '.flac', '.m4a')


def stats6(arr):
    """mean, std, min, max, kurtosis, skewness"""
    arr = arr[~np.isnan(arr)]
    if len(arr) < 2:
        return [np.nan] * 6
    return [np.mean(arr), np.std(arr), np.min(arr), np.max(arr),
            float(stats.kurtosis(arr)), float(stats.skew(arr))]


# ── Feature groups ──────────────────────────────────────

def f0_features(snd):
    pitch = snd.to_pitch_ac(pitch_floor=75, pitch_ceiling=600)
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]
    if len(f0) < 2:
        return {k: np.nan for k in
                ['f0_mean','f0_std','f0_min','f0_max','f0_kurtosis','f0_skewness','f0_std_norm']}
    m, s, mn, mx, ku, sk = stats6(f0)
    return {'f0_mean': m, 'f0_std': s, 'f0_min': mn, 'f0_max': mx,
            'f0_kurtosis': ku, 'f0_skewness': sk,
            'f0_std_norm': s / m if m > 0 else np.nan}


def intensity_features(snd):
    intensity = snd.to_intensity(minimum_pitch=75)
    vals = intensity.values.flatten()
    m, s, mn, mx, ku, sk = stats6(vals)
    return {'intensity_mean': m, 'intensity_std': s, 'intensity_min': mn,
            'intensity_max': mx, 'intensity_kurtosis': ku, 'intensity_skewness': sk}


def energy_features(y):
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=256)[0]
    m, s, mn, mx, ku, sk = stats6(rms)
    return {'energy_mean': m, 'energy_std': s, 'energy_min': mn,
            'energy_max': mx, 'energy_kurtosis': ku, 'energy_skewness': sk}


def pause_features(y, sr=16000):
    """WebRTC VAD-based pause detection — robust to background noise."""

    total_dur = len(y) / sr

    # Convert float32 → int16 PCM (required by WebRTC VAD)
    pcm = np.clip(y, -1.0, 1.0)
    pcm = (pcm * 32767).astype(np.int16)

    # VAD: aggressiveness 2 = moderate (good balance for noisy recordings)
    vad = webrtcvad.Vad(2)
    frame_ms  = 20                        # 20ms frames
    frame_len = int(sr * frame_ms / 1000) # 320 samples at 16kHz

    # Classify each 20ms frame as speech (True) or silence (False)
    labels = []
    for i in range(0, len(pcm) - frame_len, frame_len):
        try:
            labels.append(vad.is_speech(pcm[i:i+frame_len].tobytes(), sr))
        except Exception:
            labels.append(True)

    # Merge runs and filter out very short bursts/gaps
    MIN_SILENCE = 5   # minimum 5 frames (100ms) to count as a pause
    MIN_SPEECH  = 2   # minimum 2 frames  (40ms)  to count as speech

    runs = [(k, len(list(g))) for k, g in groupby(labels)]

    merged = []
    for is_speech, count in runs:
        if not is_speech and count < MIN_SILENCE:
            merged.append((True, count))
        elif is_speech and count < MIN_SPEECH:
            merged.append((False, count))
        else:
            merged.append((is_speech, count))

    # Re-merge same adjacent labels after filtering
    final = []
    for is_speech, count in merged:
        if final and final[-1][0] == is_speech:
            final[-1] = (is_speech, final[-1][1] + count)
        else:
            final.append((is_speech, count))

    # Convert frame counts → durations in seconds
    pauses, speech_segs = [], []
    for is_speech, count in final:
        dur = count * frame_ms / 1000
        (speech_segs if is_speech else pauses).append(dur)

    p   = np.array(pauses)      if pauses      else np.array([0.0])
    s   = np.array(speech_segs) if speech_segs else np.array([total_dur])
    sp  = s.sum()
    ipi = s[1:-1] if len(s) > 2 else np.array([0.0])
    n_p = len(pauses)

    return {
        'mean_pause_duration':          float(np.mean(p))      if n_p > 0 else 0.0,
        'total_speech_time':            float(sp),
        'total_pause_time':             float(p.sum())         if n_p > 0 else 0.0,
        'pause_percentage':             float(p.sum() / total_dur * 100) if n_p > 0 else 0.0,
        'silence_to_speech_ratio':      float(p.sum() / sp)   if n_p > 0 and sp > 0 else 0.0,
        'pause_variability':            float(np.std(p))       if n_p > 1 else 0.0,
        'number_of_pauses':             n_p,
        'number_of_speech_segments':    len(speech_segs),
        'max_pause_duration':           float(np.max(p))       if n_p > 0 else 0.0,
        'min_pause_duration':           float(np.min(p))       if n_p > 0 else 0.0,
        'pause_rate':                   n_p / total_dur,
        'mean_speech_segment_duration': float(np.mean(s)),
        'std_speech_segment_duration':  float(np.std(s))       if len(s) > 1 else 0.0,
        'mean_inter_pause_interval':    float(np.mean(ipi)),
        'std_inter_pause_interval':     float(np.std(ipi))     if len(ipi) > 1 else 0.0,
        'long_pause_count':             int(np.sum(p > 0.5)),
        'short_pause_count':            int(np.sum((p > 0) & (p <= 0.5))),
        'pause_duration_skewness':      float(stats.skew(p))   if len(p) > 2 else 0.0,
    }


def articulation_features(snd):
    total_dur = snd.get_total_duration()

    # Voiced duration → approximate syllable count
    pitch = snd.to_pitch_ac(pitch_floor=75, pitch_ceiling=600)
    f0 = pitch.selected_array['frequency']
    voiced_dur = np.sum(f0 > 0) * pitch.time_step
    n_syl = max(voiced_dur / 0.15, 1)  # ~150ms per syllable average

    # Speech time (excl. pauses)
    intensity = snd.to_intensity(minimum_pitch=75)
    tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
    n = call(tg, "Get number of intervals", 1)
    speech_time = sum(
        call(tg, "Get end time of interval", 1, i) - call(tg, "Get start time of interval", 1, i)
        for i in range(1, n + 1)
        if call(tg, "Get label of interval", 1, i) == "speech"
    )

    return {
        'speech_rate':           n_syl / total_dur,
        'articulation_rate':     n_syl / speech_time if speech_time > 0 else np.nan,
        'avg_syllable_duration': speech_time / n_syl if n_syl > 0 else np.nan,
    }


def formant_features(snd):
    fmt = snd.to_formant_burg(max_number_of_formants=5, maximum_formant=5500,
                               window_length=0.025, pre_emphasis_from=50)
    t = fmt.xs()
    f1 = np.array([call(fmt, "Get value at time", 1, ti, 'Hertz', 'Linear') for ti in t])
    f2 = np.array([call(fmt, "Get value at time", 2, ti, 'Hertz', 'Linear') for ti in t])
    f1, f2 = f1[~np.isnan(f1)], f2[~np.isnan(f2)]
    disp = np.mean(f2) - np.mean(f1) if len(f1) and len(f2) else np.nan
    vtl = 34300 / (2 * disp) if disp and disp > 0 else np.nan
    return {'f1_std': np.std(f1) if len(f1) > 1 else np.nan,
            'f2_std': np.std(f2) if len(f2) > 1 else np.nan,
            'vocal_tract_length': vtl}


# ── Main extraction ─────────────────────────────────────

def extract_all(filepath):
    y, _ = librosa.load(filepath, sr=16000, mono=True)
    snd   = parselmouth.Sound(filepath)
    row = {}
    row.update(f0_features(snd))
    row.update(intensity_features(snd))
    row.update(energy_features(y))
    row.update(pause_features(y))
    row.update(articulation_features(snd))
    row.update(formant_features(snd))
    return row


def load_folder(folder, label):
    rows = []
    files = [f for f in sorted(glob.glob(os.path.join(folder, '*')))
             if f.endswith(AUDIO_EXTS)]
    for fp in files:
        uid = Path(fp).stem
        print(f"  [{label}] {uid}")
        try:
            row = extract_all(fp)
            row['uid'] = uid
            row['group'] = label
            rows.append(row)
        except Exception as e:
            print(f"    ERROR: {e}")
    return rows


print("Extracting features...")
rows = load_folder(CONTROL_DIR, 'Control') + load_folder(ALZHEIMER_DIR, 'Alzheimer')
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved → {OUTPUT_CSV}  ({len(df)} rows)")


# ── Visualization ───────────────────────────────────────

FEATURE_COLS = [c for c in df.columns if c not in ('uid', 'group')]
GROUPS  = ['Control', 'Alzheimer']
COLORS  = {'Control': '#60a5fa', 'Alzheimer': '#f87171'}
os.makedirs(PLOTS_DIR, exist_ok=True)

n_cols = 6
n_rows = -(-len(FEATURE_COLS) // n_cols)   # ceiling division

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3.2, n_rows * 3.2))
fig.patch.set_facecolor('#0d1117')
axes = axes.flatten()

for i, feat in enumerate(FEATURE_COLS):
    ax = axes[i]
    ax.set_facecolor('#111827')
    ax.spines[['top', 'right']].set_visible(False)
    for spine in ax.spines.values():
        spine.set_color('#1e293b')

    data = [df[df['group'] == g][feat].dropna().values for g in GROUPS]

    bp = ax.boxplot(data, patch_artist=True, widths=0.4,
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(color='#475569'),
                    capprops=dict(color='#475569'),
                    flierprops=dict(marker='o', markerfacecolor='#475569', markersize=2))
    for patch, g in zip(bp['boxes'], GROUPS):
        patch.set_facecolor(COLORS[g])
        patch.set_alpha(0.65)

    # Individual points with jitter
    for j, (vals, g) in enumerate(zip(data, GROUPS)):
        jitter = np.random.uniform(-0.12, 0.12, len(vals))
        ax.scatter(np.full(len(vals), j + 1) + jitter, vals,
                   color=COLORS[g], alpha=0.55, s=14, zorder=5)

    # Stats annotation
    if len(data[0]) > 1 and len(data[1]) > 1:
        pooled = np.sqrt((np.var(data[0]) + np.var(data[1])) / 2)
        d = (np.mean(data[1]) - np.mean(data[0])) / pooled if pooled > 0 else 0
        _, p = stats.ttest_ind(data[0], data[1], equal_var=False)
        col = '#fbbf24' if p < 0.05 else '#94a3b8'
        ax.set_title(f"{feat}\nd={d:.2f}  p={p:.3f}", fontsize=6.5, color=col, pad=4)
    else:
        ax.set_title(feat, fontsize=6.5, color='#94a3b8', pad=4)

    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Ctrl', 'AD'], fontsize=7, color='#94a3b8')
    ax.tick_params(axis='y', labelsize=6, colors='#64748b')

for i in range(len(FEATURE_COLS), len(axes)):
    axes[i].set_visible(False)

legend = [Patch(facecolor=COLORS[g], label=g) for g in GROUPS]
fig.legend(handles=legend, loc='lower right', fontsize=9,
           facecolor='#111827', edgecolor='#1e293b', labelcolor='white')

fig.suptitle('Acoustic Biomarkers — Control vs Alzheimer (n=40)',
             fontsize=13, color='white', y=1.005)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, 'all_features.png')
plt.savefig(out, dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.close()
print(f"Plot saved → {out}")
print("\nDone.")

Extracting features...
  [Control] abgk
  [Control] ablf
  [Control] afop
  [Control] afpc
  [Control] afrv
  [Control] agok
  [Control] ahmw
  [Control] ahyz
  [Control] ajdc
  [Control] alov
  [Control] alrq
  [Control] also
  [Control] aluc
  [Control] amhy
  [Control] anpg
  [Control] aohw
  [Control] apez
  [Control] apns
  [Control] apps
  [Control] apyc
  [Control] atmc
  [Control] aulf
  [Control] avvv
  [Control] awch
  [Control] awxj
  [Control] ayqu
  [Control] ayro
  [Control] bdtr
  [Control] bguv
  [Control] bhms
  [Control] bijy
  [Control] bjzd
  [Control] bkju
  [Control] blde
  [Control] bldi
  [Control] blyu
  [Control] bmok
  [Control] bnib
  [Control] bnxv
  [Control] bpdx
  [Control] bpom
  [Control] brou
  [Control] brrd
  [Control] bufm
  [Control] bumu
  [Control] bxog
  [Control] bymf
  [Control] caur
  [Control] ccbt
  [Control] ceqd
  [Alzheimer] acqd
  [Alzheimer] aikf
  [Alzheimer] akni
  [Alzheimer] amsn
  [Alzheimer] ansa
  [Alzheimer] avdc
  [Alzheimer]

In [2]:
"""
135-Feature Acoustic Biomarker Extractor — Train/Test, CN vs AD
------------------------------------------------------------------
pip install praat-parselmouth librosa scipy pandas numpy webrtcvad-wheels

Reads metadata.csv, filters English + Control/ProbableAD/AD, finds each
audio file in TRAIN_FOLDER or TEST_FOLDER (whichever matches its split),
extracts all 135 features, and saves:

    OUTPUT/train/cn/features.csv
    OUTPUT/train/ad/features.csv
    OUTPUT/test/cn/features.csv
    OUTPUT/test/ad/features.csv

Note on libraries: Voice Quality (jitter/shimmer/HNR) uses Parselmouth,
matching the biomarker table. MFCC/Delta-MFCC use librosa, also matching
the table. Alpha Ratio, Hammarberg Index, Spectral Flux and Spectral
Slope are computed directly from the STFT using their eGeMAPS definitions
(numpy/librosa) rather than the actual openSMILE binary, since openSMILE
requires a separate external install — the formulas are the same ones
openSMILE uses internally.
"""

import os, glob
from itertools import groupby
import numpy as np
import pandas as pd
import librosa
import parselmouth
from parselmouth.praat import call
from scipy import stats
import webrtcvad
from pathlib import Path

# ── Paths ───────────────────────────────────────────────
METADATA      = r"C:\Users\arday\Downloads\metadata.csv"
TRAIN_FOLDER  = r"C:\Users\arday\Downloads\train_audios"
TEST_FOLDER   = r"C:\Users\arday\Downloads\test_audios"
OUTPUT_DIR    = r"C:\Users\arday\Desktop\full_features"
# ────────────────────────────────────────────────────────

AUDIO_EXTS = ('.wav', '.mp3', '.flac', '.m4a')


def stats6(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) < 2:
        return [np.nan] * 6
    return [np.mean(arr), np.std(arr), np.min(arr), np.max(arr),
            float(stats.kurtosis(arr)), float(stats.skew(arr))]


# ── Prosody: F0, Intensity, Energy ──────────────────────

def f0_features(pitch):
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]
    m, s, mn, mx, ku, sk = stats6(f0)
    return {'f0_mean': m, 'f0_std': s, 'f0_min': mn, 'f0_max': mx,
            'f0_kurtosis': ku, 'f0_skewness': sk,
            'f0_std_norm': s / m if m and m > 0 else np.nan}


def intensity_features(intensity):
    vals = intensity.values.flatten()
    m, s, mn, mx, ku, sk = stats6(vals)
    return {'intensity_mean': m, 'intensity_std': s, 'intensity_min': mn,
            'intensity_max': mx, 'intensity_kurtosis': ku, 'intensity_skewness': sk}


def energy_features(y):
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=256)[0]
    m, s, mn, mx, ku, sk = stats6(rms)
    return {'energy_mean': m, 'energy_std': s, 'energy_min': mn,
            'energy_max': mx, 'energy_kurtosis': ku, 'energy_skewness': sk}


# ── Pause & Fluency (WebRTC VAD) ─────────────────────────

def pause_features(y, sr=16000):
    total_dur = len(y) / sr
    pcm = (np.clip(y, -1.0, 1.0) * 32767).astype(np.int16)

    vad = webrtcvad.Vad(2)
    frame_ms, frame_len = 20, int(sr * 20 / 1000)

    labels = []
    for i in range(0, len(pcm) - frame_len, frame_len):
        try:
            labels.append(vad.is_speech(pcm[i:i+frame_len].tobytes(), sr))
        except Exception:
            labels.append(True)

    MIN_SILENCE, MIN_SPEECH = 5, 2
    runs = [(k, len(list(g))) for k, g in groupby(labels)]
    merged = []
    for is_speech, count in runs:
        if not is_speech and count < MIN_SILENCE:
            merged.append((True, count))
        elif is_speech and count < MIN_SPEECH:
            merged.append((False, count))
        else:
            merged.append((is_speech, count))

    final = []
    for is_speech, count in merged:
        if final and final[-1][0] == is_speech:
            final[-1] = (is_speech, final[-1][1] + count)
        else:
            final.append((is_speech, count))

    pauses, speech_segs = [], []
    for is_speech, count in final:
        dur = count * frame_ms / 1000
        (speech_segs if is_speech else pauses).append(dur)

    p   = np.array(pauses)      if pauses      else np.array([0.0])
    s   = np.array(speech_segs) if speech_segs else np.array([total_dur])
    sp  = s.sum()
    ipi = s[1:-1] if len(s) > 2 else np.array([0.0])
    n_p = len(pauses)

    return {
        'mean_pause_duration':          float(np.mean(p)) if n_p else 0.0,
        'total_speech_time':            float(sp),
        'total_pause_time':             float(p.sum()) if n_p else 0.0,
        'pause_percentage':             float(p.sum()/total_dur*100) if n_p else 0.0,
        'silence_to_speech_ratio':      float(p.sum()/sp) if n_p and sp>0 else 0.0,
        'pause_variability':            float(np.std(p)) if n_p>1 else 0.0,
        'number_of_pauses':             n_p,
        'number_of_speech_segments':    len(speech_segs),
        'max_pause_duration':           float(np.max(p)) if n_p else 0.0,
        'min_pause_duration':           float(np.min(p)) if n_p else 0.0,
        'pause_rate':                   n_p/total_dur,
        'mean_speech_segment_duration': float(np.mean(s)),
        'std_speech_segment_duration':  float(np.std(s)) if len(s)>1 else 0.0,
        'mean_inter_pause_interval':    float(np.mean(ipi)),
        'std_inter_pause_interval':     float(np.std(ipi)) if len(ipi)>1 else 0.0,
        'long_pause_count':             int(np.sum(p > 0.5)),
        'short_pause_count':            int(np.sum((p>0)&(p<=0.5))),
        'pause_duration_skewness':      float(stats.skew(p)) if len(p)>2 else 0.0,
    }


# ── Articulation & Resonance ──────────────────────────────

def articulation_features(snd, pitch, intensity):
    total_dur = snd.get_total_duration()
    f0 = pitch.selected_array['frequency']
    voiced_dur = np.sum(f0 > 0) * pitch.time_step
    n_syl = max(voiced_dur / 0.15, 1)

    tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
    n = call(tg, "Get number of intervals", 1)
    speech_time = sum(
        call(tg, "Get end time of interval", 1, i) - call(tg, "Get start time of interval", 1, i)
        for i in range(1, n + 1)
        if call(tg, "Get label of interval", 1, i) == "speech"
    )
    return {
        'speech_rate':           n_syl / total_dur,
        'articulation_rate':     n_syl / speech_time if speech_time > 0 else np.nan,
        'avg_syllable_duration': speech_time / n_syl if n_syl > 0 else np.nan,
    }


def formant_features(snd):
    fmt = snd.to_formant_burg(time_step=0.02, max_number_of_formants=5, maximum_formant=5500,
                               window_length=0.025, pre_emphasis_from=50)
    t = fmt.xs()
    f1 = np.array([call(fmt, "Get value at time", 1, ti, 'Hertz', 'Linear') for ti in t])
    f2 = np.array([call(fmt, "Get value at time", 2, ti, 'Hertz', 'Linear') for ti in t])
    f1, f2 = f1[~np.isnan(f1)], f2[~np.isnan(f2)]
    disp = np.mean(f2) - np.mean(f1) if len(f1) and len(f2) else np.nan
    vtl = 34300 / (2 * disp) if disp and disp > 0 else np.nan
    return {'f1_std': np.std(f1) if len(f1) > 1 else np.nan,
            'f2_std': np.std(f2) if len(f2) > 1 else np.nan,
            'vocal_tract_length': vtl}


# ── Voice Quality: Jitter, Shimmer, HNR ──────────────────

def voice_quality_features(snd, window=1.0, hop=0.5):
    total_dur = snd.get_total_duration()
    jl_vals, jd_vals, sl_vals, sa_vals = [], [], [], []

    t = 0.0
    while t + window <= total_dur:
        try:
            part = snd.extract_part(t, t + window, preserve_times=False)
            pp = call(part, "To PointProcess (periodic, cc)", 75, 600)
            if call(pp, "Get number of points") >= 4:
                jl_vals.append(call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3))
                jd_vals.append(call(pp, "Get jitter (ddp)", 0, 0, 0.0001, 0.02, 1.3))
                sl_vals.append(call([part, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6))
                sa_vals.append(call([part, pp], "Get shimmer (apq3)", 0, 0, 0.0001, 0.02, 1.3, 1.6))
        except Exception:
            pass
        t += hop

    out = {}
    for name, vals in [('jitter_local', jl_vals), ('jitter_ddp', jd_vals),
                        ('shimmer_local', sl_vals), ('shimmer_apq3', sa_vals)]:
        m, s, mn, mx, ku, sk = stats6(vals)
        out[f'{name}_mean'] = m;  out[f'{name}_std'] = s
        out[f'{name}_min']  = mn; out[f'{name}_max'] = mx
        out[f'{name}_kurtosis'] = ku; out[f'{name}_skewness'] = sk

    harmonicity = snd.to_harmonicity_cc(time_step=0.01, minimum_pitch=75)
    hnr_vals = harmonicity.values.flatten()
    hnr_vals = hnr_vals[hnr_vals > -200]
    m, s, mn, mx, ku, sk = stats6(hnr_vals)
    out['hnr_mean'] = m; out['hnr_std'] = s; out['hnr_min'] = mn
    out['hnr_max'] = mx; out['hnr_kurtosis'] = ku; out['hnr_skewness'] = sk
    return out


# ── Spectral: MFCC / Delta-MFCC (librosa) ────────────────

def mfcc_features(y, sr=16000, n_mfcc=13):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    out = {}
    for i in range(n_mfcc):
        out[f'mfcc_{i+1}_mean'] = float(np.mean(mfcc[i]))
        out[f'mfcc_{i+1}_std']  = float(np.std(mfcc[i]))
    for i in range(n_mfcc):
        out[f'delta_mfcc_{i+1}_mean'] = float(np.mean(delta[i]))
        out[f'delta_mfcc_{i+1}_std']  = float(np.std(delta[i]))
    return out


# ── Spectral Shape: Alpha Ratio, Hammarberg, Flux, Slope ─

def spectral_shape_features(y, sr=16000, n_fft=1024, hop_length=256):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    power = S ** 2

    low, high = (freqs >= 50) & (freqs < 1000), (freqs >= 1000) & (freqs <= 5000)
    e_low, e_high = power[low].sum(axis=0), power[high].sum(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        alpha_ratio = 20 * np.log10(np.where(e_high > 0, e_low / np.where(e_high > 0, e_high, 1), np.nan))

    h1, h2 = (freqs < 2000), (freqs >= 2000) & (freqs <= 5000)
    peak1, peak2 = power[h1].max(axis=0), power[h2].max(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        hammarberg = 20 * np.log10(np.where(peak2 > 0, peak1 / np.where(peak2 > 0, peak2, 1), np.nan))

    S_norm = S / (S.sum(axis=0, keepdims=True) + 1e-10)
    flux = np.sqrt(np.sum(np.diff(S_norm, axis=1) ** 2, axis=0))

    def band_slope(mask):
        x = freqs[mask]
        y = 20 * np.log10(S[mask] + 1e-10)          # shape: (k_freqs, n_frames)
        x_c = x - x.mean()
        denom = np.sum(x_c ** 2)
        y_c = y - y.mean(axis=0, keepdims=True)
        return (x_c[:, None] * y_c).sum(axis=0) / denom

    slope_0_500    = band_slope((freqs >= 0) & (freqs <= 500))
    slope_500_1500 = band_slope((freqs > 500) & (freqs <= 1500))

    out = {}
    for name, arr in [('alpha_ratio', alpha_ratio), ('hammarberg_index', hammarberg),
                       ('spectral_flux', flux), ('spectral_slope_0_500', slope_0_500),
                       ('spectral_slope_500_1500', slope_500_1500)]:
        arr = np.asarray(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        out[f'{name}_mean'] = float(np.mean(arr)) if len(arr) else np.nan
        out[f'{name}_std']  = float(np.std(arr))  if len(arr) else np.nan
    return out


# ── Combine everything ────────────────────────────────────

def extract_all(filepath):
    y, _ = librosa.load(filepath, sr=16000, mono=True)
    snd  = parselmouth.Sound(filepath)

    pitch     = snd.to_pitch_ac(pitch_floor=75, pitch_ceiling=600)
    intensity = snd.to_intensity(minimum_pitch=75)

    row = {}
    row.update(f0_features(pitch))
    row.update(intensity_features(intensity))
    row.update(energy_features(y))
    row.update(pause_features(y))
    row.update(articulation_features(snd, pitch, intensity))
    row.update(formant_features(snd))
    row.update(voice_quality_features(snd))
    row.update(mfcc_features(y))
    row.update(spectral_shape_features(y))
    return row


# ── Metadata-driven file discovery ────────────────────────

def build_index(folder):
    """Walk the folder ONCE and build a lookup dict — avoids re-scanning
    the whole directory tree for every single file (this was the main
    bottleneck: os.walk was being run once per row)."""
    exact, prefix_list = {}, []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(AUDIO_EXTS):
                path = os.path.join(root, f)
                exact[Path(f).stem] = path
                prefix_list.append((f, path))
    return exact, prefix_list


def find_audio(uid, exact, prefix_list):
    if uid in exact:
        return exact[uid]
    for fname, path in prefix_list:
        if fname.startswith(uid):
            return path
    return None


def process_group(df_rows, exact, prefix_list, split, group_label):
    out_dir = os.path.join(OUTPUT_DIR, split, group_label.lower())
    os.makedirs(out_dir, exist_ok=True)
    rows = []
    for _, r in df_rows.iterrows():
        uid = r['uid']
        path = find_audio(uid, exact, prefix_list)
        if not path:
            print(f"  [!] {split}/{group_label} — not found: {uid}")
            continue
        print(f"  [{split}/{group_label}] {uid}")
        try:
            feat = extract_all(path)
            feat['uid'] = uid
            feat['group'] = group_label
            rows.append(feat)
        except Exception as e:
            print(f"    ERROR ({uid}): {e}")

    df_out = pd.DataFrame(rows)
    df_out.to_csv(os.path.join(out_dir, 'features.csv'), index=False)
    print(f"  Saved {len(df_out)} rows → {out_dir}/features.csv\n")
    return df_out


# ── Main ───────────────────────────────────────────────────

meta = pd.read_csv(METADATA)
meta_eng = meta[meta['language'] == 'english']

for split, folder in [('train', TRAIN_FOLDER), ('test', TEST_FOLDER)]:
    print(f"\n{'='*50}\n{split.upper()}\n{'='*50}")
    split_df = meta_eng[meta_eng['split'] == split]

    cn_df = split_df[split_df['diagnosis'] == 'Control']
    ad_df = split_df[split_df['diagnosis'].isin(['ProbableAD', 'AD'])]

    print(f"Pool — CN: {len(cn_df)}  AD: {len(ad_df)}")
    print(f"Indexing {folder} ...")
    exact, prefix_list = build_index(folder)
    print(f"  {len(prefix_list)} audio files found")

    process_group(cn_df, exact, prefix_list, split, 'CN')
    process_group(ad_df, exact, prefix_list, split, 'AD')

print("\nDone.")


TRAIN
Pool — CN: 760  AD: 423
Indexing C:\Users\arday\Downloads\train_audios ...
  1646 audio files found
  [train/CN] abgk
  [train/CN] ablf
  [train/CN] afop
  [train/CN] afpc
  [train/CN] afrv
  [train/CN] agok
  [train/CN] ahmw
  [train/CN] ahyz
  [train/CN] aijc
  [train/CN] ajdc
  [train/CN] alov
  [train/CN] alrq
  [train/CN] also
  [train/CN] aluc
  [train/CN] amhy
  [train/CN] anpg
  [train/CN] aohw
  [train/CN] apez
  [train/CN] apns
  [train/CN] apps
  [train/CN] apyc
  [train/CN] atmc
  [train/CN] aulf
  [train/CN] avvv
  [train/CN] awch
  [train/CN] awxj
  [train/CN] ayqu
  [train/CN] ayro
  [train/CN] bajg
  [train/CN] bdtr
  [train/CN] bfvg
  [train/CN] bguv
  [train/CN] bhms
  [train/CN] bijy
  [train/CN] bjzd
  [train/CN] bkju
  [train/CN] blde
  [train/CN] bldi
  [train/CN] blyu
  [train/CN] bmok
  [train/CN] bnib
  [train/CN] bnxv
  [train/CN] bpdx
  [train/CN] bpom
  [train/CN] brou
  [train/CN] brpt
  [train/CN] brrd
  [train/CN] bufm
  [train/CN] bumu
  [train/CN

C:\Users\arday\AppData\Local\Temp\ipykernel_18388\3210188414.py:142: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  'pause_duration_skewness':      float(stats.skew(p)) if len(p)>2 else 0.0,


  [train/AD] xeql
  [train/AD] xfag
  [train/AD] xgem
  [train/AD] xmkn
  [train/AD] xnjt
  [train/AD] xoct
  [train/AD] xqvq
  [train/AD] xrem
  [train/AD] xsdx
  [train/AD] xsrd
  [train/AD] xtck
  [train/AD] xvhq
  [train/AD] xwva
  [train/AD] xxbk
  [train/AD] xygd
  [train/AD] ybxe
  [train/AD] ycrj
  [train/AD] yfsu
  [train/AD] ykgr
  [train/AD] ykqp
  [train/AD] ylsp
  [train/AD] yrrz
  [train/AD] ytfb
  [train/AD] yusb
  [train/AD] ywwq
  [train/AD] yzwd
  [train/AD] zaxk
  [train/AD] zcyg
  [train/AD] zdpe
  [train/AD] zgcl
  [train/AD] zhot
  [train/AD] zjvs
  [train/AD] zjvt
  [train/AD] zkbm
  [train/AD] zkuk
  [train/AD] zljk
  [train/AD] zllx
  [train/AD] zmkc
  [train/AD] zobv
  [train/AD] zpir
  [train/AD] zriz
  [train/AD] zsbf
  [train/AD] zvhj
  Saved 423 rows → C:\Users\arday\Desktop\full_features\train\ad/features.csv


TEST
Pool — CN: 184  AD: 114
Indexing C:\Users\arday\Downloads\test_audios ...
  412 audio files found
  [test/CN] aazd
  [test/CN] acdn
  [test/C

C:\Users\arday\AppData\Local\Temp\ipykernel_18388\3210188414.py:142: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  'pause_duration_skewness':      float(stats.skew(p)) if len(p)>2 else 0.0,


  [test/CN] uahb
  [test/CN] udld
  [test/CN] udvs
  [test/CN] ugal
  [test/CN] ustr
  [test/CN] uvnq
  [test/CN] uvqo
  [test/CN] uwvu
  [test/CN] vacv
  [test/CN] vbdd
  [test/CN] vgkt
  [test/CN] vgtm
  [test/CN] vkmd
  [test/CN] vnup
  [test/CN] vpfs
  [test/CN] vtky
  [test/CN] vzaf
  [test/CN] wain
  [test/CN] wgcz
  [test/CN] wgdr
  [test/CN] wmek
  [test/CN] wpti
  [test/CN] xicq
  [test/CN] xjme
  [test/CN] xlmp
  [test/CN] xsds
  [test/CN] ydew
  [test/CN] ydin
  [test/CN] ygom
  [test/CN] ynsn
  [test/CN] ypmg
  [test/CN] yrhi
  [test/CN] ysor
  [test/CN] ywit
  [test/CN] ywse
  [test/CN] yxpb
  [test/CN] yxsy
  [test/CN] yzjr
  [test/CN] zbdh
  [test/CN] zceh
  [test/CN] zcey
  [test/CN] zkeq
  [test/CN] zmez
  [test/CN] zprd
  [test/CN] zrkf
  [test/CN] zrzh
  [test/CN] ztlq
  [test/CN] zwye
  [test/CN] zycr
  Saved 184 rows → C:\Users\arday\Desktop\full_features\test\cn/features.csv

  [test/AD] agon
  [test/AD] ajef
  [test/AD] amdz
  [test/AD] axqe
  [test/AD] azfh
  [t

In [3]:
"""
Step 1 — Merge, Clean, and Scale Feature Data
------------------------------------------------
pip install pandas numpy scikit-learn joblib

Merges train/cn + train/ad into one labeled train set,
test/cn + test/ad into one labeled test set. Fills missing
values with the train median, applies RobustScaler (fit on
train only), and saves the processed CSVs + the fitted scaler.
"""

import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib

# ── Paths ───────────────────────────────────────────────
TRAIN_CN_CSV = r"C:\Users\arday\Desktop\full_features\train\cn\features.csv"
TRAIN_AD_CSV = r"C:\Users\arday\Desktop\full_features\train\ad\features.csv"
TEST_CN_CSV  = r"C:\Users\arday\Desktop\full_features\test\cn\features.csv"
TEST_AD_CSV  = r"C:\Users\arday\Desktop\full_features\test\ad\features.csv"
OUTPUT_DIR   = r"C:\Users\arday\Desktop\full_features\processed"
# ────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load and label: CN=0, AD=1
train_cn = pd.read_csv(TRAIN_CN_CSV); train_cn['label'] = 0
train_ad = pd.read_csv(TRAIN_AD_CSV); train_ad['label'] = 1
test_cn  = pd.read_csv(TEST_CN_CSV);  test_cn['label']  = 0
test_ad  = pd.read_csv(TEST_AD_CSV);  test_ad['label']  = 1

train_df = pd.concat([train_cn, train_ad], ignore_index=True)
test_df  = pd.concat([test_cn, test_ad], ignore_index=True)

print(f"Train: {len(train_df)} rows  (CN={len(train_cn)}, AD={len(train_ad)})")
print(f"Test:  {len(test_df)} rows  (CN={len(test_cn)}, AD={len(test_ad)})")

meta_cols = ['uid', 'group', 'label']
feature_cols = [c for c in train_df.columns if c not in meta_cols]

# Drop features with too much missing data in train (>30%)
missing_frac = train_df[feature_cols].isna().mean()
drop_cols = missing_frac[missing_frac > 0.3].index.tolist()
if drop_cols:
    print(f"\nDropping {len(drop_cols)} features with >30% missing in train:")
    print(f"  {drop_cols}")
feature_cols = [c for c in feature_cols if c not in drop_cols]

# Fill remaining NaNs with the TRAIN median (applied to both sets)
train_median = train_df[feature_cols].median()
train_df[feature_cols] = train_df[feature_cols].fillna(train_median)
test_df[feature_cols]  = test_df[feature_cols].fillna(train_median)

# Robust scaling — fit on train only, apply to both
scaler = RobustScaler()
train_scaled = scaler.fit_transform(train_df[feature_cols])
test_scaled  = scaler.transform(test_df[feature_cols])

train_out = pd.DataFrame(train_scaled, columns=feature_cols)
train_out['uid']   = train_df['uid'].values
train_out['label'] = train_df['label'].values

test_out = pd.DataFrame(test_scaled, columns=feature_cols)
test_out['uid']   = test_df['uid'].values
test_out['label'] = test_df['label'].values

train_out.to_csv(os.path.join(OUTPUT_DIR, 'train_processed.csv'), index=False)
test_out.to_csv(os.path.join(OUTPUT_DIR, 'test_processed.csv'), index=False)
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler.pkl'))
joblib.dump(feature_cols, os.path.join(OUTPUT_DIR, 'feature_cols.pkl'))

print(f"\nFinal feature count: {len(feature_cols)}")
print(f"Saved → {OUTPUT_DIR}\\train_processed.csv")
print(f"Saved → {OUTPUT_DIR}\\test_processed.csv")
print(f"Saved → {OUTPUT_DIR}\\scaler.pkl  (reuse this in later steps)")
print(f"Saved → {OUTPUT_DIR}\\feature_cols.pkl  (final feature list)")

C:\Users\arday\AppData\Local\Temp\ipykernel_18388\3433852374.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_cn = pd.read_csv(TRAIN_CN_CSV); train_cn['label'] = 0
C:\Users\arday\AppData\Local\Temp\ipykernel_18388\3433852374.py:30: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_ad = pd.read_csv(TRAIN_AD_CSV); train_ad['label'] = 1
C:\Users\arday\AppData\Local\Temp\ipykernel_18388\3433852374.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

Train: 1183 rows  (CN=760, AD=423)
Test:  298 rows  (CN=184, AD=114)

Final feature count: 135
Saved → C:\Users\arday\Desktop\full_features\processed\train_processed.csv
Saved → C:\Users\arday\Desktop\full_features\processed\test_processed.csv
Saved → C:\Users\arday\Desktop\full_features\processed\scaler.pkl  (reuse this in later steps)
Saved → C:\Users\arday\Desktop\full_features\processed\feature_cols.pkl  (final feature list)


In [4]:
"""
Step 2 — Statistical Analysis: Cohen's d and p-value for all features
------------------------------------------------------------------------
pip install pandas numpy scipy

Computes effect size (Cohen's d) and significance (Welch t-test) for
every feature in the processed train set, comparing CN vs AD. Saves a
sorted table used later for feature selection (Step 3).
"""

import os
import pandas as pd
import numpy as np
from scipy import stats

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV = r"C:\Users\arday\Desktop\full_features\processed\train_processed.csv"
OUTPUT_DIR = r"C:\Users\arday\Desktop\full_features\processed"
# ────────────────────────────────────────────────────────

df = pd.read_csv(TRAIN_CSV)
feature_cols = [c for c in df.columns if c not in ('uid', 'label')]

cn = df[df['label'] == 0]
ad = df[df['label'] == 1]
print(f"CN: {len(cn)}   AD: {len(ad)}   Features: {len(feature_cols)}\n")

results = []
for f in feature_cols:
    c = cn[f].dropna().values
    a = ad[f].dropna().values
    if len(c) < 2 or len(a) < 2:
        continue

    pooled = np.sqrt((np.var(c) + np.var(a)) / 2)
    d = (np.mean(a) - np.mean(c)) / pooled if pooled > 0 else 0.0
    _, p = stats.ttest_ind(c, a, equal_var=False)

    results.append({
        'feature': f,
        'd': d,
        'abs_d': abs(d),
        'p': p,
        'significant': p < 0.05,
        'mean_cn': np.mean(c),
        'mean_ad': np.mean(a),
    })

res_df = pd.DataFrame(results).sort_values('abs_d', ascending=False).reset_index(drop=True)

out_path = os.path.join(OUTPUT_DIR, 'feature_stats.csv')
res_df.to_csv(out_path, index=False)

print(f"{'feature':<32}{'d':>8}{'p':>10}{'sig':>6}")
print("=" * 58)
for _, r in res_df.head(30).iterrows():
    flag = '*' if r['significant'] else ''
    print(f"{r['feature']:<32}{r['d']:>8.2f}{r['p']:>10.4f}{flag:>6}")

n_sig = res_df['significant'].sum()
n_large = (res_df['abs_d'] >= 0.5).sum()
n_medium = ((res_df['abs_d'] >= 0.3) & (res_df['abs_d'] < 0.5)).sum()

print(f"\nTotal features:        {len(res_df)}")
print(f"Significant (p<0.05):  {n_sig}")
print(f"Large effect (|d|>=0.5): {n_large}")
print(f"Medium effect (0.3-0.5): {n_medium}")
print(f"\nSaved → {out_path}")

CN: 760   AD: 423   Features: 135

feature                                d         p   sig
spectral_slope_0_500_mean           0.40    0.0000     *
mfcc_12_mean                       -0.33    0.0000     *
hammarberg_index_mean              -0.32    0.0000     *
spectral_flux_std                  -0.29    0.0000     *
delta_mfcc_6_std                    0.29    0.0000     *
mfcc_13_mean                       -0.29    0.0000     *
mfcc_7_mean                        -0.29    0.0000     *
mfcc_7_std                          0.29    0.0000     *
delta_mfcc_11_std                   0.29    0.0000     *
hammarberg_index_std                0.29    0.0000     *
mfcc_9_std                          0.29    0.0000     *
mfcc_11_std                         0.28    0.0000     *
delta_mfcc_8_std                    0.28    0.0000     *
delta_mfcc_10_std                   0.28    0.0000     *
spectral_flux_mean                 -0.28    0.0000     *
mfcc_10_std                         0.28    0.0000   

In [5]:
"""
Step 3 — Feature Selection: Effect Size Filtering + Correlation Pruning
---------------------------------------------------------------------------
pip install pandas numpy

Starts from the Step 2 stats table, keeps features with |d| >= D_THRESHOLD,
then removes redundant features: for any pair with |correlation| > CORR_THRESHOLD,
drops the one with the smaller effect size. Saves the final feature list.
"""

import os
import pandas as pd
import numpy as np

# ── Paths & thresholds ───────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features\processed\train_processed.csv"
STATS_CSV    = r"C:\Users\arday\Desktop\full_features\processed\feature_stats.csv"
OUTPUT_DIR   = r"C:\Users\arday\Desktop\full_features\processed"
D_THRESHOLD    = 0.2   # minimum |Cohen's d| to be considered a candidate
CORR_THRESHOLD = 0.85  # above this, features are considered redundant
# ─────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
stats_df = pd.read_csv(STATS_CSV)

candidates = stats_df[stats_df['abs_d'] >= D_THRESHOLD].sort_values('abs_d', ascending=False)
cand_features = candidates['feature'].tolist()
print(f"Candidates with |d| >= {D_THRESHOLD}: {len(cand_features)}")

# Correlation matrix among candidates only
corr = train_df[cand_features].corr().abs()

# Greedy pruning: walk features from strongest d to weakest,
# drop any later feature that's too correlated with one already kept
d_lookup = dict(zip(candidates['feature'], candidates['abs_d']))
kept = []
dropped = []

for f in cand_features:  # already sorted by |d| descending
    if f in dropped:
        continue
    too_correlated_with_kept = any(corr.loc[f, k] > CORR_THRESHOLD for k in kept)
    if too_correlated_with_kept:
        dropped.append(f)
        continue
    kept.append(f)
    # mark any remaining candidate too correlated with this one as dropped
    for other in cand_features:
        if other != f and other not in kept and other not in dropped:
            if corr.loc[f, other] > CORR_THRESHOLD:
                dropped.append(other)

print(f"\nFinal feature count after correlation pruning: {len(kept)}")
print(f"Dropped as redundant: {len(dropped)}\n")

print(f"{'feature':<32}{'d':>8}")
print("=" * 40)
for f in kept:
    print(f"{f:<32}{d_lookup[f]:>8.3f}")

pd.Series(kept).to_csv(os.path.join(OUTPUT_DIR, 'selected_features.csv'), index=False, header=['feature'])
print(f"\nSaved → {OUTPUT_DIR}\\selected_features.csv")

Candidates with |d| >= 0.2: 42

Final feature count after correlation pruning: 29
Dropped as redundant: 13

feature                                d
spectral_slope_0_500_mean          0.397
mfcc_12_mean                       0.328
hammarberg_index_mean              0.323
spectral_flux_std                  0.294
delta_mfcc_6_std                   0.293
mfcc_13_mean                       0.291
mfcc_7_mean                        0.290
mfcc_7_std                         0.288
delta_mfcc_11_std                  0.287
hammarberg_index_std               0.287
mfcc_9_std                         0.286
delta_mfcc_8_std                   0.284
mfcc_10_std                        0.276
delta_mfcc_13_std                  0.272
mfcc_11_mean                       0.272
mfcc_5_mean                        0.259
hnr_min                            0.254
f0_std                             0.243
f1_std                             0.241
mfcc_9_mean                        0.235
intensity_std                  

In [6]:
"""
Step 4 — Baseline Models with 5-Fold Cross-Validation
----------------------------------------------------------
pip install pandas numpy scikit-learn

Trains Logistic Regression, SVM (linear + RBF), and Random Forest
on the selected features using 5-fold stratified cross-validation.
Reports mean AUC, sensitivity, specificity, and balanced accuracy
for each model so the best one can be chosen for Step 5.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features\processed\train_processed.csv"
FEATURES_CSV = r"C:\Users\arday\Desktop\full_features\processed\selected_features.csv"
N_FOLDS      = 5
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()

X = train_df[features].values
y = train_df['label'].values

print(f"Samples: {len(y)}   Features: {len(features)}   CN={sum(y==0)}  AD={sum(y==1)}\n")

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'SVM (linear)':        SVC(kernel='linear', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'SVM (rbf)':           SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE),
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = []
for name, model in models.items():
    aucs, sens, spec, bal_accs = [], [], [], []

    for train_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        proba = model.predict_proba(X_val)[:, 1]
        pred  = (proba >= 0.5).astype(int)

        aucs.append(roc_auc_score(y_val, proba))
        tn, fp, fn, tp = confusion_matrix(y_val, pred, labels=[0, 1]).ravel()
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)  # sensitivity: AD recall
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)  # specificity: CN recall
        bal_accs.append(balanced_accuracy_score(y_val, pred))

    results.append({
        'model': name,
        'AUC': np.mean(aucs),
        'AUC_std': np.std(aucs),
        'Sensitivity': np.mean(sens),
        'Specificity': np.mean(spec),
        'Balanced Acc': np.mean(bal_accs),
    })
    print(f"  Done: {name}")

res_df = pd.DataFrame(results).sort_values('AUC', ascending=False)

print(f"\n{'Model':<22}{'AUC':>8}{'±std':>7}{'Sens':>8}{'Spec':>8}{'BalAcc':>9}")
print("=" * 62)
for _, r in res_df.iterrows():
    print(f"{r['model']:<22}{r['AUC']:>8.3f}{r['AUC_std']:>7.3f}"
          f"{r['Sensitivity']:>8.3f}{r['Specificity']:>8.3f}{r['Balanced Acc']:>9.3f}")

Samples: 1183   Features: 29   CN=760  AD=423

  Done: Logistic Regression
  Done: SVM (linear)
  Done: SVM (rbf)
  Done: Random Forest

Model                      AUC   ±std    Sens    Spec   BalAcc
Random Forest            0.629  0.055   0.250   0.909    0.580
SVM (rbf)                0.627  0.049   0.222   0.879    0.551
SVM (linear)             0.616  0.061   0.062   0.938    0.500
Logistic Regression      0.597  0.065   0.565   0.604    0.584


In [7]:
"""
Step 4b — Threshold Tuning (Youden's J)
----------------------------------------------------------
pip install pandas numpy scikit-learn

AUC measures ranking ability across ALL thresholds, but sensitivity/
specificity only look at one cutoff (0.5 by default). This script
collects out-of-fold probabilities across 5-fold CV, then finds the
threshold that maximizes Youden's J = sensitivity + specificity - 1
instead of using 0.5. Reports both the default and tuned results
side by side.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score, roc_curve

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features\processed\train_processed.csv"
FEATURES_CSV = r"C:\Users\arday\Desktop\full_features\processed\selected_features.csv"
N_FOLDS      = 5
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()

X = train_df[features].values
y = train_df['label'].values

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'SVM (linear)':        SVC(kernel='linear', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'SVM (rbf)':           SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE),
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)


def metrics_at_threshold(y_true, proba, thresh):
    pred = (proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal  = balanced_accuracy_score(y_true, pred)
    return sens, spec, bal


results = []
for name, model in models.items():
    # Out-of-fold probabilities: each sample predicted exactly once,
    # always by a model that never saw it during training.
    oof_proba = np.zeros(len(y))
    for train_idx, val_idx in skf.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        oof_proba[val_idx] = model.predict_proba(X[val_idx])[:, 1]

    auc = roc_auc_score(y, oof_proba)

    fpr, tpr, thresholds = roc_curve(y, oof_proba)
    best_idx = np.argmax(tpr - fpr)          # Youden's J = tpr - fpr
    best_thresh = thresholds[best_idx]

    sens_05, spec_05, bal_05 = metrics_at_threshold(y, oof_proba, 0.5)
    sens_opt, spec_opt, bal_opt = metrics_at_threshold(y, oof_proba, best_thresh)

    results.append({
        'model': name, 'AUC': auc, 'best_thresh': best_thresh,
        'sens_0.5': sens_05, 'spec_0.5': spec_05, 'bal_0.5': bal_05,
        'sens_opt': sens_opt, 'spec_opt': spec_opt, 'bal_opt': bal_opt,
    })
    print(f"  Done: {name}  (optimal threshold = {best_thresh:.3f})")

res_df = pd.DataFrame(results).sort_values('bal_opt', ascending=False)

print(f"\n{'Model':<22}{'AUC':>7}{'Thr':>7} | {'Sens.5':>8}{'Spec.5':>8}{'Bal.5':>8} | {'SensOpt':>8}{'SpecOpt':>8}{'BalOpt':>8}")
print("=" * 96)
for _, r in res_df.iterrows():
    print(f"{r['model']:<22}{r['AUC']:>7.3f}{r['best_thresh']:>7.3f} | "
          f"{r['sens_0.5']:>8.3f}{r['spec_0.5']:>8.3f}{r['bal_0.5']:>8.3f} | "
          f"{r['sens_opt']:>8.3f}{r['spec_opt']:>8.3f}{r['bal_opt']:>8.3f}")

  Done: Logistic Regression  (optimal threshold = 0.462)
  Done: SVM (linear)  (optimal threshold = 0.381)
  Done: SVM (rbf)  (optimal threshold = 0.394)
  Done: Random Forest  (optimal threshold = 0.433)

Model                     AUC    Thr |   Sens.5  Spec.5   Bal.5 |  SensOpt SpecOpt  BalOpt
SVM (linear)            0.607  0.381 |    0.061   0.938   0.500 |    0.546   0.693   0.620
SVM (rbf)               0.623  0.394 |    0.222   0.879   0.551 |    0.520   0.717   0.619
Random Forest           0.627  0.433 |    0.251   0.909   0.580 |    0.397   0.803   0.600
Logistic Regression     0.592  0.462 |    0.565   0.604   0.584 |    0.662   0.536   0.599


In [9]:
"""
Step 4c — Hyperparameter Tuning + XGBoost (Nested Cross-Validation)
------------------------------------------------------------------------
pip install pandas numpy scikit-learn xgboost

Nested CV: an INNER GridSearchCV (3-fold) picks the best hyperparameters
within each OUTER fold (5-fold), and the resulting best estimator is
evaluated on the held-out outer fold. This avoids the optimistic bias
of picking hyperparameters and evaluating on the same folds. Out-of-fold
probabilities are then used to find the optimal threshold (Youden's J),
same approach as Step 4b.
"""

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score, roc_curve
from xgboost import XGBClassifier

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features\processed\train_processed.csv"
FEATURES_CSV = r"C:\Users\arday\Desktop\full_features\processed\selected_features.csv"
N_OUTER      = 5
N_INNER      = 3
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()

X = train_df[features].values
y = train_df['label'].values

pos_weight = (y == 0).sum() / (y == 1).sum()  # for XGBoost class imbalance
print(f"Samples: {len(y)}  Features: {len(features)}  CN={sum(y==0)}  AD={sum(y==1)}  pos_weight={pos_weight:.2f}\n")

model_grids = {
    'Logistic Regression': (
        LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1, 10]}
    ),
    'SVM (linear)': (
        SVC(kernel='linear', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': [0.01, 0.1, 1, 10]}
    ),
    'SVM (rbf)': (
        SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
        {'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.1]}
    ),
    'Random Forest': (
        RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
        {'n_estimators': [200, 400], 'max_depth': [None, 5, 10], 'min_samples_leaf': [1, 3, 5]}
    ),
    'XGBoost': (
        XGBClassifier(eval_metric='logloss', random_state=RANDOM_STATE),
        {'n_estimators': [200, 400], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1],
         'scale_pos_weight': [1, pos_weight]}
    ),
}


def metrics_at_threshold(y_true, proba, thresh):
    pred = (proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal  = balanced_accuracy_score(y_true, pred)
    return sens, spec, bal


outer_skf = StratifiedKFold(n_splits=N_OUTER, shuffle=True, random_state=RANDOM_STATE)

results = []
for name, (base_model, grid) in model_grids.items():
    print(f"Tuning {name} ...")
    oof_proba = np.zeros(len(y))

    for fold_i, (train_idx, val_idx) in enumerate(outer_skf.split(X, y), 1):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]

        inner_skf = StratifiedKFold(n_splits=N_INNER, shuffle=True, random_state=RANDOM_STATE)
        search = GridSearchCV(base_model, grid, scoring='roc_auc', cv=inner_skf, n_jobs=-1)
        search.fit(X_tr, y_tr)

        oof_proba[val_idx] = search.best_estimator_.predict_proba(X_val)[:, 1]
        print(f"  Fold {fold_i}: best params = {search.best_params_}")

    auc = roc_auc_score(y, oof_proba)
    fpr, tpr, thresholds = roc_curve(y, oof_proba)
    best_thresh = thresholds[np.argmax(tpr - fpr)]

    sens_05, spec_05, bal_05 = metrics_at_threshold(y, oof_proba, 0.5)
    sens_opt, spec_opt, bal_opt = metrics_at_threshold(y, oof_proba, best_thresh)

    results.append({
        'model': name, 'AUC': auc, 'best_thresh': best_thresh,
        'sens_0.5': sens_05, 'spec_0.5': spec_05, 'bal_0.5': bal_05,
        'sens_opt': sens_opt, 'spec_opt': spec_opt, 'bal_opt': bal_opt,
    })
    print(f"  → AUC={auc:.3f}  BalAcc(opt)={bal_opt:.3f}\n")

res_df = pd.DataFrame(results).sort_values('bal_opt', ascending=False)

print(f"\n{'Model':<22}{'AUC':>7}{'Thr':>7} | {'Sens.5':>8}{'Spec.5':>8}{'Bal.5':>8} | {'SensOpt':>8}{'SpecOpt':>8}{'BalOpt':>8}")
print("=" * 96)
for _, r in res_df.iterrows():
    print(f"{r['model']:<22}{r['AUC']:>7.3f}{r['best_thresh']:>7.3f} | "
          f"{r['sens_0.5']:>8.3f}{r['spec_0.5']:>8.3f}{r['bal_0.5']:>8.3f} | "
          f"{r['sens_opt']:>8.3f}{r['spec_opt']:>8.3f}{r['bal_opt']:>8.3f}")

out_path = os.path.join(os.path.dirname(TRAIN_CSV), 'tuned_model_results.csv')
res_df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")

Samples: 1183  Features: 29  CN=760  AD=423  pos_weight=1.80

Tuning Logistic Regression ...
  Fold 1: best params = {'C': 0.01}
  Fold 2: best params = {'C': 0.01}
  Fold 3: best params = {'C': 0.01}
  Fold 4: best params = {'C': 0.01}
  Fold 5: best params = {'C': 0.01}
  → AUC=0.606  BalAcc(opt)=0.591

Tuning SVM (linear) ...
  Fold 1: best params = {'C': 0.01}
  Fold 2: best params = {'C': 0.01}
  Fold 3: best params = {'C': 0.01}
  Fold 4: best params = {'C': 1}
  Fold 5: best params = {'C': 0.01}
  → AUC=0.606  BalAcc(opt)=0.598

Tuning SVM (rbf) ...
  Fold 1: best params = {'C': 1, 'gamma': 0.01}
  Fold 2: best params = {'C': 1, 'gamma': 0.01}
  Fold 3: best params = {'C': 1, 'gamma': 'scale'}
  Fold 4: best params = {'C': 10, 'gamma': 0.01}
  Fold 5: best params = {'C': 1, 'gamma': 'scale'}
  → AUC=0.621  BalAcc(opt)=0.616

Tuning Random Forest ...
  Fold 1: best params = {'max_depth': 5, 'min_samples_leaf': 3, 'n_estimators': 200}
  Fold 2: best params = {'max_depth': 5, 'min_

In [10]:
"""
Step 4d — Ensemble: Averaging + Stacking (default, untuned models)
------------------------------------------------------------------------
pip install pandas numpy scikit-learn xgboost

Hyperparameter tuning gave no real improvement (Step 4c), so we go back
to default settings for all models. This script combines them two ways:

  1. Simple averaging — mean of each model's out-of-fold probability
  2. Stacking — a meta Logistic Regression trained on the out-of-fold
     probabilities of the base models (classic OOF-stacking, no leakage
     since the meta-model only ever sees probabilities that were
     generated by base models that never saw that sample)

All base models share the SAME 5-fold split so their OOF probabilities
line up sample-for-sample, which is required for both ensemble methods.
"""

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score, roc_curve
from xgboost import XGBClassifier

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features\processed\train_processed.csv"
FEATURES_CSV = r"C:\Users\arday\Desktop\full_features\processed\selected_features.csv"
N_FOLDS      = 5
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()

X = train_df[features].values
y = train_df['label'].values

pos_weight = (y == 0).sum() / (y == 1).sum()  # standard imbalance formula, not searched

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'SVM (linear)':        SVC(kernel='linear', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'SVM (rbf)':           SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE),
    'XGBoost':             XGBClassifier(eval_metric='logloss', scale_pos_weight=pos_weight, random_state=RANDOM_STATE),
}


def metrics_at_threshold(y_true, proba, thresh):
    pred = (proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal  = balanced_accuracy_score(y_true, pred)
    return sens, spec, bal


def summarize(name, y_true, proba):
    auc = roc_auc_score(y_true, proba)
    fpr, tpr, thresholds = roc_curve(y_true, proba)
    best_thresh = thresholds[np.argmax(tpr - fpr)]
    sens, spec, bal = metrics_at_threshold(y_true, proba, best_thresh)
    return {'model': name, 'AUC': auc, 'best_thresh': best_thresh,
            'sens_opt': sens, 'spec_opt': spec, 'bal_opt': bal}


skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# ── Step 1: collect OOF probabilities from every base model ──
oof = {}
for name, model in models.items():
    print(f"Running {name} ...")
    proba = np.zeros(len(y))
    for train_idx, val_idx in skf.split(X, y):
        model.fit(X[train_idx], y[train_idx])
        proba[val_idx] = model.predict_proba(X[val_idx])[:, 1]
    oof[name] = proba

oof_matrix = np.column_stack([oof[n] for n in models])

# ── Step 2: simple averaging ──────────────────────────────
avg_proba = oof_matrix.mean(axis=1)

# ── Step 3: stacking — meta Logistic Regression on OOF matrix ──
meta_oof = np.zeros(len(y))
for train_idx, val_idx in skf.split(oof_matrix, y):
    meta = LogisticRegression(class_weight='balanced')
    meta.fit(oof_matrix[train_idx], y[train_idx])
    meta_oof[val_idx] = meta.predict_proba(oof_matrix[val_idx])[:, 1]

# ── Results ────────────────────────────────────────────────
results = [summarize(name, y, oof[name]) for name in models]
results.append(summarize('Ensemble (average)', y, avg_proba))
results.append(summarize('Ensemble (stacking)', y, meta_oof))

res_df = pd.DataFrame(results).sort_values('bal_opt', ascending=False)

print(f"\n{'Model':<24}{'AUC':>7}{'Thr':>7}{'SensOpt':>9}{'SpecOpt':>9}{'BalOpt':>9}")
print("=" * 65)
for _, r in res_df.iterrows():
    print(f"{r['model']:<24}{r['AUC']:>7.3f}{r['best_thresh']:>7.3f}"
          f"{r['sens_opt']:>9.3f}{r['spec_opt']:>9.3f}{r['bal_opt']:>9.3f}")

out_path = os.path.join(os.path.dirname(TRAIN_CSV), 'ensemble_results.csv')
res_df.to_csv(out_path, index=False)
print(f"\nSaved → {out_path}")

Running Logistic Regression ...
Running SVM (linear) ...
Running SVM (rbf) ...
Running Random Forest ...
Running XGBoost ...

Model                       AUC    Thr  SensOpt  SpecOpt   BalOpt
SVM (linear)              0.607  0.381    0.546    0.693    0.620
SVM (rbf)                 0.623  0.394    0.520    0.717    0.619
Ensemble (average)        0.626  0.381    0.579    0.649    0.614
Ensemble (stacking)       0.622  0.533    0.478    0.743    0.610
Random Forest             0.627  0.433    0.397    0.803    0.600
Logistic Regression       0.592  0.462    0.662    0.536    0.599
XGBoost                   0.607  0.212    0.598    0.578    0.588

Saved → C:\Users\arday\Desktop\full_features\processed\ensemble_results.csv


In [11]:
"""
Step 5 — Final Model: Train on Full Train Set, Evaluate Once on Test Set
------------------------------------------------------------------------
pip install pandas numpy scikit-learn joblib

Trains SVM (rbf) — the best-performing model from Step 4 — on the ENTIRE
train set. The classification threshold is chosen using out-of-fold
predictions on the train set only (test is never touched for this),
then applied as a FIXED threshold when evaluating on the test set.
This is the only point where the test set is used — for final
reporting, not for model or threshold selection.
"""

import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score, roc_curve

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features\processed\train_processed.csv"
TEST_CSV     = r"C:\Users\arday\Desktop\full_features\processed\test_processed.csv"
FEATURES_CSV = r"C:\Users\arday\Desktop\full_features\processed\selected_features.csv"
OUTPUT_DIR   = r"C:\Users\arday\Desktop\full_features\processed"
N_FOLDS      = 5
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()

X_train, y_train = train_df[features].values, train_df['label'].values
X_test,  y_test  = test_df[features].values,  test_df['label'].values

print(f"Train: {len(y_train)}  (CN={sum(y_train==0)}, AD={sum(y_train==1)})")
print(f"Test:  {len(y_test)}  (CN={sum(y_test==0)}, AD={sum(y_test==1)})\n")


def metrics_at_threshold(y_true, proba, thresh):
    pred = (proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal  = balanced_accuracy_score(y_true, pred)
    return sens, spec, bal


# ── Step A: find the threshold using TRAIN-only out-of-fold CV ──
print("Finding threshold via 5-fold CV on train set...")
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(y_train))

for train_idx, val_idx in skf.split(X_train, y_train):
    model = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE)
    model.fit(X_train[train_idx], y_train[train_idx])
    oof_proba[val_idx] = model.predict_proba(X_train[val_idx])[:, 1]

fpr, tpr, thresholds = roc_curve(y_train, oof_proba)
threshold = thresholds[np.argmax(tpr - fpr)]
train_auc = roc_auc_score(y_train, oof_proba)
print(f"  Train CV AUC:     {train_auc:.3f}")
print(f"  Chosen threshold: {threshold:.3f}  (from train CV only, test untouched)\n")

# ── Step B: train the FINAL model on the ENTIRE train set ──
print("Training final model on full train set...")
final_model = SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE)
final_model.fit(X_train, y_train)

# ── Step C: evaluate ONCE on the test set ──
test_proba = final_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_proba)
sens, spec, bal = metrics_at_threshold(y_test, test_proba, threshold)

print(f"\n{'='*50}")
print("FINAL TEST SET RESULTS (evaluated once)")
print(f"{'='*50}")
print(f"  AUC:                {test_auc:.3f}")
print(f"  Threshold used:     {threshold:.3f}")
print(f"  Sensitivity:        {sens:.3f}")
print(f"  Specificity:        {spec:.3f}")
print(f"  Balanced Accuracy:  {bal:.3f}")

joblib.dump(final_model, os.path.join(OUTPUT_DIR, 'final_model_svm_rbf.pkl'))
joblib.dump(threshold, os.path.join(OUTPUT_DIR, 'final_threshold.pkl'))
print(f"\nSaved → {OUTPUT_DIR}\\final_model_svm_rbf.pkl")
print(f"Saved → {OUTPUT_DIR}\\final_threshold.pkl")

Train: 1183  (CN=760, AD=423)
Test:  298  (CN=184, AD=114)

Finding threshold via 5-fold CV on train set...
  Train CV AUC:     0.623
  Chosen threshold: 0.394  (from train CV only, test untouched)

Training final model on full train set...

FINAL TEST SET RESULTS (evaluated once)
  AUC:                0.664
  Threshold used:     0.394
  Sensitivity:        0.535
  Specificity:        0.745
  Balanced Accuracy:  0.640

Saved → C:\Users\arday\Desktop\full_features\processed\final_model_svm_rbf.pkl
Saved → C:\Users\arday\Desktop\full_features\processed\final_threshold.pkl


In [12]:
"""
Step 7 — Interpretability: SHAP + Permutation Importance
------------------------------------------------------------------------
pip install pandas numpy scikit-learn shap matplotlib joblib

Explains which of the 29 selected features actually drive the model's
predictions, using two independent methods that should broadly agree
if the findings are trustworthy:

  1. Permutation importance — shuffle one feature at a time on the
     TEST set and measure how much AUC drops. Model-agnostic, exact.
  2. SHAP (KernelExplainer) — attributes each prediction's probability
     to individual feature contributions. Works for any model,
     including non-linear ones like SVM (rbf), unlike TreeExplainer
     or LinearExplainer which only work for specific model types.
"""

import os
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV      = r"C:\Users\arday\Desktop\full_features\processed\train_processed.csv"
TEST_CSV       = r"C:\Users\arday\Desktop\full_features\processed\test_processed.csv"
FEATURES_CSV   = r"C:\Users\arday\Desktop\full_features\processed\selected_features.csv"
MODEL_PATH     = r"C:\Users\arday\Desktop\full_features\processed\final_model_svm_rbf.pkl"
OUTPUT_DIR     = r"C:\Users\arday\Desktop\full_features\processed"
N_SHAP_SAMPLES = 150     # how many test rows to explain with SHAP (speed vs completeness)
RANDOM_STATE   = 42
# ────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()
model    = joblib.load(MODEL_PATH)

X_train = train_df[features].values
X_test  = test_df[features].values
y_test  = test_df['label'].values

# ── 1. Permutation importance (test set, AUC drop) ──
print("Computing permutation importance...")
perm = permutation_importance(
    model, X_test, y_test,
    scoring='roc_auc', n_repeats=30, random_state=RANDOM_STATE, n_jobs=-1
)
perm_df = pd.DataFrame({
    'feature': features,
    'perm_importance_mean': perm.importances_mean,
    'perm_importance_std':  perm.importances_std,
}).sort_values('perm_importance_mean', ascending=False)

# ── 2. SHAP (KernelExplainer — SVM rbf is non-linear, needs model-agnostic method) ──
print("Computing SHAP values (this can take a few minutes)...")
background = shap.kmeans(X_train, 50)   # compact reference set, not the full train set
rng = np.random.RandomState(RANDOM_STATE)
explain_idx = rng.choice(len(X_test), size=min(N_SHAP_SAMPLES, len(X_test)), replace=False)
X_explain = X_test[explain_idx]

predict_ad_proba = lambda x: model.predict_proba(x)[:, 1]
explainer = shap.KernelExplainer(predict_ad_proba, background)
shap_values = explainer.shap_values(X_explain, nsamples='auto')

shap_df = pd.DataFrame({
    'feature': features,
    'mean_abs_shap': np.abs(shap_values).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)

# ── Combine and print ──
combined = perm_df.merge(shap_df, on='feature').sort_values('mean_abs_shap', ascending=False)

print(f"\n{'feature':<28}{'SHAP':>10}{'PermImp':>10}")
print("=" * 48)
for _, r in combined.head(20).iterrows():
    print(f"{r['feature']:<28}{r['mean_abs_shap']:>10.4f}{r['perm_importance_mean']:>10.4f}")

combined.to_csv(os.path.join(OUTPUT_DIR, 'feature_importance.csv'), index=False)


# ── Plots ──
def dark_barplot(df, value_col, title, filename, top_n=20):
    top = df.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 7))
    fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#111827')
    ax.barh(top['feature'], top[value_col], color='#60a5fa')
    ax.set_title(title, color='white', fontsize=12)
    ax.tick_params(colors='#94a3b8', labelsize=8)
    for spine in ax.spines.values():
        spine.set_color('#1e293b')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150, facecolor='#0d1117')
    plt.close()


dark_barplot(shap_df, 'mean_abs_shap', 'Top Features — Mean |SHAP value|', 'shap_importance.png')
dark_barplot(perm_df, 'perm_importance_mean', 'Top Features — Permutation Importance (AUC drop)', 'permutation_importance.png')

print(f"\nSaved → {OUTPUT_DIR}\\feature_importance.csv")
print(f"Saved → {OUTPUT_DIR}\\shap_importance.png")
print(f"Saved → {OUTPUT_DIR}\\permutation_importance.png")

C:\Users\arday\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Computing permutation importance...
Computing SHAP values (this can take a few minutes)...


100%|██████████| 150/150 [22:20<00:00,  8.93s/it]



feature                           SHAP   PermImp
spectral_slope_0_500_mean       0.0145   -0.0006
intensity_mean                  0.0138    0.0030
hnr_min                         0.0129    0.0078
shimmer_local_skewness          0.0121    0.0006
hammarberg_index_mean           0.0097    0.0039
spectral_flux_std               0.0077    0.0055
mfcc_12_mean                    0.0071    0.0007
mfcc_9_mean                     0.0068    0.0075
shimmer_local_std               0.0063    0.0096
mfcc_10_std                     0.0063   -0.0005
mfcc_13_mean                    0.0058    0.0028
delta_mfcc_8_std                0.0055   -0.0025
delta_mfcc_13_std               0.0050    0.0009
mfcc_5_mean                     0.0046    0.0030
jitter_local_max                0.0039   -0.0033
mfcc_11_mean                    0.0032    0.0016
jitter_local_mean               0.0030    0.0082
hammarberg_index_std            0.0028    0.0021
f0_std                          0.0026    0.0017
mfcc_10_mean       

In [ ]:
import os
import glob
import librosa
import numpy as np
import soundfile as sf
from scipy.signal import butter, lfilter

# --- CONFIGURATION (Dynamic Relative Paths) ---

# 1. Find the exact folder where this Python script is currently located
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# 2. Set Input and Output paths relative to the script's location
INPUT_DIR = os.path.join(BASE_DIR, "..", "data", "train_audios")
OUTPUT_DIR = os.path.join(BASE_DIR, "..", "data", "processed_audios")

# --- FILTER SETTINGS ---
LOWCUT_HZ = 75.0      # Eliminates low-frequency mechanical rumble
HIGHCUT_HZ = 8000.0   # Eliminates high-frequency electronic hiss
FILTER_ORDER = 4      # Order of the Butterworth filter (steepness)


def butter_bandpass(lowcut, highcut, fs, order=4):
    """
    Creates the Butterworth band-pass filter coefficients.
    """
    nyq = 0.5 * fs # Nyquist frequency
    low = lowcut / nyq
    
    # Ensure highcut doesn't exceed Nyquist frequency for low-quality audios
    high = min(highcut / nyq, 0.999) 
    
    b, a = butter(order, [low, high], btype='band')
    return b, a


def apply_bandpass_filter(data, fs, lowcut, highcut, order=4):
    """
    Applies the band-pass filter to the audio data.
    """
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y_filtered = lfilter(b, a, data)
    return y_filtered


def preprocess_audio(file_path, output_path):
    """
    Applies Band-Pass filter and then Peak Normalization.
    NO Resampling is done (keeps original sample rate).
    """
    # 1. Load audio with its ORIGINAL sample rate (sr=None)
    y, sr = librosa.load(file_path, sr=None)
    
    # 2. Apply Butterworth Band-Pass Filter (using the original sr)
    y_filtered = apply_bandpass_filter(y, sr, LOWCUT_HZ, HIGHCUT_HZ, order=FILTER_ORDER)
    
    # 3. Apply Peak Normalization
    max_amplitude = np.max(np.abs(y_filtered))
    if max_amplitude > 0:
        y_normalized = y_filtered / max_amplitude
    else:
        y_normalized = y_filtered  # Skip if the audio file is completely silent
    
    # 4. Save the final processed file using the ORIGINAL sample rate
    sf.write(output_path, y_normalized, sr)


def main():
    """
    Main pipeline function to process all audio files.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    wav_files = glob.glob(os.path.join(INPUT_DIR, "*.wav"))
    mp3_files = glob.glob(os.path.join(INPUT_DIR, "*.mp3"))
    
    audio_files = wav_files + mp3_files

    if not audio_files:
        print(f"No audio files (.wav or .mp3) found in: {INPUT_DIR}")
        return

    print(f"Starting processing for {len(audio_files)} files...")
    print(f"Steps: 1. Keep Original SR -> 2. Band-Pass Filter -> 3. Peak Normalization\n")

    for file_path in audio_files:
        file_name = os.path.basename(file_path)
        file_base_name = os.path.splitext(file_name)[0]
        output_file_name = f"{file_base_name}.wav"
        output_path = os.path.join(OUTPUT_DIR, output_file_name)

        try:
            preprocess_audio(file_path, output_path)
            print(f"SUCCESS - {file_name} -> filtered & normalized (Original SR)")
        except Exception as e:
            print(f"ERROR - {file_name} failed: {e}")

    print("\nAll audio files processed successfully.")


if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import librosa
import soundfile as sf
from scipy.signal import butter, lfilter

# --- CONFIGURATION (Dynamic Relative Paths) ---

# 1. Find the exact folder where this Python script is currently located
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# 2. Set Input and Output paths relative to the script's location
INPUT_DIR = os.path.join(BASE_DIR, "..", "data", "train_audios")
OUTPUT_DIR = os.path.join(BASE_DIR, "..", "data", "processed_audios")

TARGET_SR = 16000

# --- FILTER SETTINGS ---
LOWCUT_HZ = 75.0      # Eliminates low-frequency mechanical rumble
HIGHCUT_HZ = 8000.0   # Eliminates high-frequency electronic hiss
FILTER_ORDER = 4      # Order of the Butterworth filter (steepness)


def butter_bandpass(lowcut, highcut, fs, order=4):
    """
    Creates the Butterworth band-pass filter coefficients.
    """
    nyq = 0.5 * fs # Nyquist frequency
    low = lowcut / nyq
    
    # Ensure highcut doesn't exceed Nyquist frequency
    high = min(highcut / nyq, 0.999) 
    
    b, a = butter(order, [low, high], btype='band')
    return b, a


def apply_bandpass_filter(data, fs, lowcut, highcut, order=4):
    """
    Applies the band-pass filter to the audio data.
    """
    b, a = butter_bandpass(lowcut, highcut, fs, order=order)
    y_filtered = lfilter(b, a, data)
    return y_filtered


def preprocess_audio(file_path, output_path):
    """
    Applies ONLY the Band-Pass filter to the audio file.
    No trimming, no DC offset removal.
    """
    # 1. Load audio and resample to target frequency
    y, sr = librosa.load(file_path, sr=TARGET_SR)
    
    # 2. Apply ONLY Butterworth Band-Pass Filter (75 Hz - 8 kHz)
    y_filtered = apply_bandpass_filter(y, TARGET_SR, LOWCUT_HZ, HIGHCUT_HZ, order=FILTER_ORDER)
    
    # Save the filtered file
    sf.write(output_path, y_filtered, TARGET_SR)


def main():
    """
    Main pipeline function to process all audio files in the input directory.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    wav_files = glob.glob(os.path.join(INPUT_DIR, "*.wav"))
    mp3_files = glob.glob(os.path.join(INPUT_DIR, "*.mp3"))
    
    audio_files = wav_files + mp3_files

    if not audio_files:
        print(f"No audio files (.wav or .mp3) found in: {INPUT_DIR}")
        return

    print(f"Starting filtering for {len(audio_files)} files...")
    print(f"Applying ONLY Band-Pass Filter: {LOWCUT_HZ}Hz - {HIGHCUT_HZ}Hz\n")

    for file_path in audio_files:
        file_name = os.path.basename(file_path)
        file_base_name = os.path.splitext(file_name)[0]
        output_file_name = f"{file_base_name}.wav"
        output_path = os.path.join(OUTPUT_DIR, output_file_name)

        try:
            preprocess_audio(file_path, output_path)
            print(f"SUCCESS - {file_name} -> filtered & saved")
        except Exception as e:
            print(f"ERROR - {file_name} failed: {e}")

    print("\nAll audio files filtered successfully.")


if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import librosa
import numpy as np
import soundfile as sf

# --- CONFIGURATION (Dynamic Relative Paths) ---

# 1. Find the exact folder where this Python script is currently located
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# 2. Set Input and Output paths relative to the script's location
INPUT_DIR = os.path.join(BASE_DIR, "..", "data", "train_audios")
OUTPUT_DIR = os.path.join(BASE_DIR, "..", "data", "processed_audios")

TARGET_SR = 16000

def preprocess_audio(file_path, output_path):
    """
    Applies ONLY Peak Normalization to the audio file.
    No filtering, no trimming, no DC offset removal.
    Scales the audio so its maximum amplitude reaches 1.0.
    """
    # 1. Load audio and resample to target frequency
    y, sr = librosa.load(file_path, sr=TARGET_SR)
    
    # 2. Apply Peak Normalization
    max_amplitude = np.max(np.abs(y))
    if max_amplitude > 0:
        y_normalized = y / max_amplitude
    else:
        y_normalized = y  # Skip if the audio file is completely silent
    
    # Save the normalized file
    sf.write(output_path, y_normalized, TARGET_SR)

def main():
    """
    Main pipeline function to process all audio files in the input directory.
    """
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    wav_files = glob.glob(os.path.join(INPUT_DIR, "*.wav"))
    mp3_files = glob.glob(os.path.join(INPUT_DIR, "*.mp3"))
    
    audio_files = wav_files + mp3_files

    if not audio_files:
        print(f"No audio files (.wav or .mp3) found in: {INPUT_DIR}")
        return

    print(f"Starting normalization for {len(audio_files)} files...\n")

    for file_path in audio_files:
        file_name = os.path.basename(file_path)
        file_base_name = os.path.splitext(file_name)[0]
        output_file_name = f"{file_base_name}.wav"
        output_path = os.path.join(OUTPUT_DIR, output_file_name)

        try:
            preprocess_audio(file_path, output_path)
            print(f"SUCCESS - {file_name} -> normalized & saved")
        except Exception as e:
            print(f"ERROR - {file_name} failed: {e}")

    print("\nAll audio files normalized successfully.")

if __name__ == "__main__":
    main()